# Pipeline Medallón STASYS — Google Colab

Este notebook lee los archivos compatibles de una carpeta de Google Drive, perfila sus columnas, conserva los originales y guarda las capas limpias y el modelo dimensional en S3. Está pensado para ejecutarse **en Colab**, no desde la carpeta local de VS Code.

## Ejecución
1. Copia los CSV/datasets a la carpeta de Drive que se indicará en la celda 2 y comparte esa carpeta con la cuenta que ejecuta Colab.
2. Ejecuta las celdas en orden y proporciona las credenciales AWS requeridas.
3. El resultado queda bajo `raw/`, `silver/cleaned/` y `gold/star_schema/`; el manifiesto registra rutas, conteos y hashes.
4. Para que otros usuarios lean los objetos S3, el equipo debe concederles permisos IAM de lectura al bucket.

## Modelo de datos
Se genera `fact_registros.parquet`, una tabla unificada con claves de dimensiones y columnas de origen `src_*`, además de un hecho detallado por dataset. Los hechos mantienen el grano original: no se cruzan consultas, atenciones y liquidaciones entre sí, porque eso multiplicaría registros y produciría métricas incorrectas. Las relaciones de IPRESS cifradas no se infieren; las claves no resueltas usan `0` y las filas se conservan.

> El perfil automático ayuda con datasets futuros, pero las nuevas relaciones de negocio requieren una regla explícita y validación del significado de las claves y medidas.

**Nota:** los archivos contienen datos sanitarios. Esta versión no incluye muestras de valores en los reportes; compartir lectura en S3 también hace accesibles esos datos a los usuarios autorizados por IAM.

# Instalacion de dependencias - Configuracion de entorno

In [1]:
# CELDA 01 - INSTALAR DEPENDENCIAS EN COLAB
# Entradas admitidas: CSV/TXT/TSV, Excel XLS/XLSX y JSON.
%pip -q install pandas boto3 google-auth google-api-python-client openpyxl xlrd

print("✓ Entorno Colab preparado")

✓ Entorno Colab preparado


In [3]:

# CELDA 02 - CONFIGURAR DRIVE Y S3
# Los usuarios que vayan a leer resultados deberán tener IAM de lectura en el bucket.
from getpass import getpass
from google.colab import auth as colab_auth
from google.auth import default as google_auth_default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import boto3

AWS_ACCESS_KEY_ID = getpass("AWS Access Key ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS Secret Access Key: ").strip()
AWS_SESSION_TOKEN = getpass("AWS Session Token (Enter si no tienes): ").strip() or None
AWS_REGION = "us-east-1"
S3_BUCKET = input("S3_BUCKET [staysis-renipress-equipo02-2026]: ").strip() or "staysis-renipress-equipo02-2026"
DRIVE_FOLDER_ID = input("ID de carpeta Drive [carpeta compartida STASYS]: ").strip() or "1NSCDPLZlJXPJsddy-D7G64IHHJnR-jwO"

session_kwargs = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region_name": AWS_REGION,
}
if AWS_SESSION_TOKEN:
    session_kwargs["aws_session_token"] = AWS_SESSION_TOKEN

s3 = boto3.client("s3", **session_kwargs)
s3.head_bucket(Bucket=S3_BUCKET)
print(f"✓ S3 conectado: {S3_BUCKET}")

colab_auth.authenticate_user()
creds, _ = google_auth_default()
drive = build("drive", "v3", credentials=creds, cache_discovery=False)
print("✓ Google Drive autenticado")

AWS Access Key ID: ··········
AWS Secret Access Key: ··········
AWS Session Token (Enter si no tienes): ··········
S3_BUCKET [staysis-renipress-equipo02-2026]: 
ID de carpeta Drive [carpeta compartida STASYS]: 
✓ S3 conectado: staysis-renipress-equipo02-2026
✓ Google Drive autenticado


In [25]:

# ============================================================
# CELDA 03 - FUNCIONES COMUNES DEL PIPELINE STASYS
# Entrada: archivos CSV/TXT/TSV, Excel y JSON
# Salida: utilidades para RAW, SILVER y GOLD
# ============================================================

import csv
import hashlib
import json
import re
import unicodedata
import uuid
import codecs
import itertools

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import pandas as pd
import numpy as np

# Identificador único de ejecución
RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:6]
)

DATE_PATH = datetime.now(timezone.utc).strftime("%Y/%m/%d")
CHUNK_SIZE = 100_000

NULL_TOKENS = {"", "n/a", "na", "null", "none", "nan", "-"}

SUPPORTED_EXTENSIONS = {
    ".csv", ".txt", ".tsv", ".xlsx", ".xls",
    ".json", ".jsonl", ".ndjson"
}


# ------------------------------------------------------------
# 1. NORMALIZACIÓN DE NOMBRES
# ------------------------------------------------------------

def slug(value):
    """Convierte nombres en identificadores seguros."""

    value = unicodedata.normalize(
        "NFKD", str(value)
    ).encode("ascii", "ignore").decode("ascii")

    return (
        re.sub(r"[^A-Za-z0-9_-]+", "_", value.strip())
        .strip("_").lower()
        or "dataset"
    )


def norm_col(value):
    """Normaliza encabezados sin tildes ni espacios."""

    value = unicodedata.normalize(
        "NFKD", str(value).strip().upper()
    ).encode("ascii", "ignore").decode("ascii")

    return (
        re.sub(r"[^A-Z0-9]+", "_", value).strip("_")
        or "COLUMNA"
    )


def unique_columns(columns):
    """Evita encabezados duplicados después de normalizarlos."""

    counts = Counter()
    result = []

    for column in columns:
        base = norm_col(column)
        counts[base] += 1

        result.append(
            base if counts[base] == 1
            else f"{base}_{counts[base]}"
        )

    return result


# ------------------------------------------------------------
# 2. LIMPIEZA BÁSICA CONSERVADORA
# ------------------------------------------------------------

def clean_chunk(frame):
    """
    Limpia espacios y tokens de nulos.
    No elimina filas ni columnas.
    """

    frame = frame.copy()

    for column in frame.columns:
        values = frame[column].astype("string").str.strip()

        frame[column] = values.mask(
            values.str.lower().isin(NULL_TOKENS),
            pd.NA
        )

    return frame


# ------------------------------------------------------------
# 3. DETECCIÓN DE CODIFICACIÓN Y SEPARADOR
# ------------------------------------------------------------

def detect_csv_settings(path):
    """Detecta codificación y delimitador de un archivo de texto."""

    path = Path(path)

    with open(path, "rb") as stream:
        sample = stream.read(128_000)

    try:
        decoder = codecs.getincrementaldecoder("utf-8-sig")("strict")

        with open(path, "rb") as stream:
            for block in iter(
                lambda: stream.read(8 * 1024 * 1024), b""
            ):
                decoder.decode(block, final=False)

            decoder.decode(b"", final=True)

        encoding = "utf-8-sig"

    except UnicodeDecodeError:
        encoding = "cp1252"

    text = sample.decode(encoding, errors="replace")
    lines = text.splitlines()
    first_line = lines[0] if lines else ""

    if (
        path.suffix.lower() == ".tsv"
        or first_line.count("\t") >
        max(
            first_line.count(","),
            first_line.count(";"),
            first_line.count("|")
        )
    ):
        delimiter = "\t"

    else:
        try:
            delimiter = csv.Sniffer().sniff(
                text[:20_000],
                delimiters=",;|"
            ).delimiter

        except csv.Error:
            delimiter = max(
                [",", ";", "|"],
                key=first_line.count
            )

    return encoding, delimiter


# ------------------------------------------------------------
# 4. LECTURA DE ARCHIVOS POR BLOQUES
# ------------------------------------------------------------

def iter_file_chunks(path, chunk_size=CHUNK_SIZE):
    """
    Lee archivos en bloques.
    Los códigos se mantienen como texto para evitar perder
    ceros iniciales.
    """

    path = Path(path)
    ext = path.suffix.lower()

    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    if path.stat().st_size == 0:
        raise ValueError(f"El archivo está vacío: {path.name}")

    if ext in {".csv", ".txt", ".tsv"}:
        encoding, delimiter = detect_csv_settings(path)

        reader = pd.read_csv(
            path,
            sep=delimiter,
            encoding=encoding,
            encoding_errors="replace",
            dtype="string",
            chunksize=chunk_size,
            keep_default_na=False,
            on_bad_lines="error"
        )

        for chunk in reader:
            chunk.columns = unique_columns(chunk.columns)
            yield clean_chunk(chunk)

        return

    if ext == ".xlsx":
        frame = pd.read_excel(
            path, dtype="string", engine="openpyxl"
        )

    elif ext == ".xls":
        frame = pd.read_excel(
            path, dtype="string", engine="xlrd"
        )

    elif ext in {".jsonl", ".ndjson"}:
        frame = pd.read_json(path, lines=True, dtype=False)

    elif ext == ".json":
        frame = pd.read_json(path, dtype=False)

    else:
        raise ValueError(f"Formato no admitido: {ext}")

    frame.columns = unique_columns(frame.columns)
    frame = clean_chunk(frame)

    for start in range(0, len(frame), chunk_size):
        yield frame.iloc[start:start + chunk_size].copy()


# ------------------------------------------------------------
# 5. CLASIFICACIÓN DE DATASETS
# ------------------------------------------------------------

def classify_dataset(filename, columns):
    """
    Clasifica los datasets STASYS.
    La farmacia nueva se identifica antes de revisar TEDEF.
    """

    name = norm_col(Path(filename).stem)
    cols = {norm_col(column) for column in columns}

    # Farmacia privada - septiembre de 2026
    pharmacy_name = (
        "PCM_BD_FARMACIA_TEDEF_2026_SEPTIEMBRE" in name
    )

    pharmacy_columns = {
        "VCODIAFAS_FAC",
        "VCODIPRESS_FAC",
        "VTDOCPAGO_FAC",
        "VNDOCPAGO_FAC",
        "VCORRPRESTA_ATE",
        "VCORRPRODFARATE_FAR",
        "VTPROD_FAR_003",
        "VMONTPRODFAR_FAR",
        "DFECRECEP_PQ"
    }

    if pharmacy_name or pharmacy_columns.issubset(cols):
        return "farmacia"

    # RENIPRESS
    if (
        {"COD_IPRESS", "UBIGEO", "NOMBRE", "INSTITUCION"}.issubset(cols)
        or "RENIPRESS" in name
    ):
        return "renipress"

    # C1 - Producción asistencial
    if (
        {"ANHO", "MES", "CO_IPRESS", "NRO_TOTAL_ATENCIONES"}.issubset(cols)
        or "CONSULTAC1" in name
        or re.search(r"(^|_)C1(_|$)", name)
    ):
        return "c1"

    # Consultas y denuncias
    if (
        {"FE_REGISTRO", "DE_TIPO_SOLICITUD", "CO_INSTITUCION"}.issubset(cols)
        or "CONSULTAS_DENUNCIAS" in name
    ):
        return "consultas_denuncias"

    # Liquidaciones EPS
    if (
        {
            "FECHAINICIOPRESTACION",
            "COD_UBIGEO",
            "TOTALLIQUIDADOSIN_IGV"
        }.issubset(cols)
        or "PRES_LIQ_EPS" in name
        or re.search(r"(^|_)EPS(_|$)", name)
    ):
        return "eps"

    # Dataset TEDEF antiguo: regla deliberadamente específica
    if (
        {
            "FECHA_PRESTACION",
            "UBIGEO_IPRESS",
            "TOTAL_LIQUIDADO_IAFAS"
        }.issubset(cols)
        or "PCM_BD_ATENCIONES_TEDEF" in name
    ):
        return "tedef"

    return "generico"


# ------------------------------------------------------------
# 6. FECHAS, UBIGEO E IPRESS
# ------------------------------------------------------------

DATE_COLUMNS = [
    "FECHA", "FECHA_PRESTACION", "FE_REGISTRO",
    "FECHALIQUIDACION", "FECHAINICIOPRESTACION",
    "FECHA_FACTURA", "FECHA_REGISTRO",
    "DFECRECEP_PQ"
]

GEO_COLUMNS = [
    "UBIGEO", "UBIGEO_IPRESS", "COD_UBIGEO",
    "DE_UBIGEO_INST", "UBIGEO_CODIGO"
]

IPRESS_COLUMNS = [
    "CO_IPRESS", "COD_IPRESS", "CODIGO_IPRESS",
    "VCODIPRESS_FAC"
]


def first_present(columns, candidates):
    return next(
        (name for name in candidates if name in columns),
        None
    )


def parse_date_values(frame, role):
    """Devuelve fechas normalizadas para el perfilado de GOLD."""

    result = pd.Series(
        pd.NaT,
        index=frame.index,
        dtype="datetime64[ns]"
    )

    if role == "c1" and {"ANHO", "MES"}.issubset(frame.columns):
        year = pd.to_numeric(frame["ANHO"], errors="coerce")
        month = pd.to_numeric(frame["MES"], errors="coerce")

        valid = year.notna() & month.between(1, 12)

        if valid.any():
            values = (
                year[valid].astype(int).astype(str).str.zfill(4)
                + "-"
                + month[valid].astype(int).astype(str).str.zfill(2)
                + "-01"
            )

            result.loc[valid] = pd.to_datetime(
                values, errors="coerce"
            ).to_numpy()

        return result

    role_date = {
        "tedef": "FECHA_PRESTACION",
        "eps": "FECHAINICIOPRESTACION",
        "consultas_denuncias": "FE_REGISTRO",
        "renipress": "INICIO_ACTIVIDAD",
        "farmacia": "DFECRECEP_PQ"
    }.get(role)

    col = (
        role_date
        if role_date in frame.columns
        else first_present(frame.columns, DATE_COLUMNS)
    )

    if col is None:
        col = next(
            (
                c for c in frame.columns
                if "FECHA" in c or "DATE" in c
            ),
            None
        )

    if col is None:
        return result

    values = (
        frame[col].astype("string").str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )

    if role == "tedef" and col == "FECHA_PRESTACION":
        return pd.to_datetime(
            values,
            format="%Y%m%d",
            errors="coerce"
        )

    parsed = pd.to_datetime(
        values,
        format="mixed",
        errors="coerce",
        dayfirst=True
    )

    try:
        return parsed.dt.tz_localize(None)
    except (AttributeError, TypeError):
        return pd.to_datetime(
            values, errors="coerce", dayfirst=True
        )


def normalize_code(value, width=None):
    """Normaliza un código sin convertirlo a un número."""

    if pd.isna(value):
        return None

    text = re.sub(r"\.0+$", "", str(value).strip())

    if not text or text.lower() in NULL_TOKENS:
        return None

    if width and text.isdigit():
        return text.zfill(width)

    return text


def get_geo_column(columns, role):
    specific = {
        "renipress": "UBIGEO",
        "c1": "UBIGEO",
        "tedef": "UBIGEO_IPRESS",
        "eps": "COD_UBIGEO",
        "consultas_denuncias": "DE_UBIGEO_INST",
        "farmacia": "UBIGEO"
    }

    if role in specific:
        column = specific[role]
        return column if column in columns else None

    return first_present(columns, GEO_COLUMNS)


def get_ipress_column(columns, role):
    if role not in {"renipress", "c1", "generico", "farmacia"}:
        return None

    candidates = [
        c for c in IPRESS_COLUMNS
        if c in columns
        and not any(
            token in c
            for token in ("CRIPTO", "CRYPTO", "HASH")
        )
    ]

    return candidates[0] if candidates else None


# ------------------------------------------------------------
# 7. PERFILADO DE TIPOS
# ------------------------------------------------------------

def infer_profile_type(column_name, values):
    """
    Inferencia orientativa.
    No reemplaza el diccionario de datos oficial.
    """

    column = norm_col(column_name)

    sample = (
        pd.Series(values, dtype="string")
        .dropna()
        .head(5000)
    )

    if sample.empty:
        return "vacío"

    pharmacy_ids = {
        "VCODIAFAS_FAC",
        "VCODIPRESS_FAC",
        "VTDOCPAGO_FAC",
        "VNDOCPAGO_FAC",
        "VCORRPRESTA_ATE",
        "VCORRPRODFARATE_FAR",
        "VCODCATALOGO_FAR_020",
        "VCODPROD_FAR_020",
        "VCODIGEMID_FAR_020",
        "VCODCATALOGO_FAR_003",
        "VCODPROD_FAR_003",
        "UBIGEO"
    }

    if column in pharmacy_ids:
        return "código/identificador (texto)"

    if (
        column.startswith((
            "COD", "CO_", "TIPO", "SEXO", "CATEGORIA",
            "UBIGEO", "DOC", "CIE"
        ))
        or column == "ID"
        or column.endswith("_ID")
        or any(
            token in column
            for token in ("CRIPTO", "CRYPTO", "HASH")
        )
    ):
        return "código/categoría (texto)"

    if "FECHA" in column or "DATE" in column or column == "DFECRECEP_PQ":
        dates = pd.to_datetime(
            sample,
            errors="coerce",
            dayfirst=True,
            format="mixed"
        )

        if dates.notna().mean() >= 0.85:
            return "fecha (inferida)"

    numeric = pd.to_numeric(
        sample.str.replace(",", ".", regex=False),
        errors="coerce"
    )

    if numeric.notna().mean() >= 0.85:
        return "numérico (inferido)"

    return "texto/categoría"


# ------------------------------------------------------------
# 8. INTEGRIDAD SHA-256
# ------------------------------------------------------------

def sha256_file(path):
    """Calcula el hash SHA-256 de un archivo por bloques."""

    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(8 * 1024 * 1024), b""
        ):
            digest.update(block)

    return digest.hexdigest()


print("✓ Funciones comunes de STASYS cargadas")
print(f"✓ RUN_ID: {RUN_ID}")
print(f"✓ Bloques CSV: {CHUNK_SIZE:,} filas")
print("✓ Farmacia se distingue del TEDEF antiguo")


✓ Funciones comunes de STASYS cargadas
✓ RUN_ID: 20261027T151544Z_1870aa
✓ Bloques CSV: 100,000 filas
✓ Farmacia se distingue del TEDEF antiguo


# Seleccion de Datasets en Drive, Local

In [27]:

# ============================================================
# CELDA 04 - SELECCIÓN DE DATASETS STASYS
# Opciones:
# 1. Subir archivos desde la PC
# 2. Elegir archivos de Google Drive
# 3. Procesar los 5 datasets principales de Google Drive
#
# Dataset nuevo: Farmacia privada - septiembre 2026
# Reemplaza al antiguo dataset de atenciones TEDEF.
# ============================================================

from pathlib import Path
from getpass import getpass
from google.colab import files as colab_files
from google.colab import auth as colab_auth
from google.auth import default as google_auth_default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
import re

# ------------------------------------------------------------
# 0. RECONSTRUIR LA CONEXIÓN CORRECTA CON GOOGLE DRIVE API
# ------------------------------------------------------------
# Esto evita el error:
# AttributeError: module 'google.colab.drive' has no attribute 'files'

try:
    creds, _ = google_auth_default()
    drive = build(
        "drive",
        "v3",
        credentials=creds,
        cache_discovery=False
    )

    # Comprobación de que drive es un servicio de Drive API v3
    drive.files().list(
        pageSize=1,
        fields="files(id,name)"
    ).execute()

    print("✓ Google Drive API v3 conectada correctamente")

except Exception as exc:
    raise RuntimeError(
        "No se pudo conectar con Google Drive API. "
        "Verifica que hayas ejecutado la celda 02 y autorizado "
        "tu cuenta de Google. Detalle: " + str(exc)
    )


# ------------------------------------------------------------
# 1. DATASETS PRINCIPALES DE STASYS
# ------------------------------------------------------------
# La farmacia se identifica por su nombre exacto.
# No clasificar todos los archivos que contienen 'TEDEF' como
# farmacia, porque pueden existir archivos TEDEF antiguos.

EXPECTED_FILES = {
    "renipress": [
        "RENIPRESS_2026_v6.csv",
        "renipress_2026_v6 (1).csv"
    ],
    "c1": [
        "ConsultaC1_Produccion_Asistencial_2015_v1.csv"
    ],
    "consultas_denuncias": [
        "CONSULTAS_DENUNCIAS_2026_v1.csv"
    ],
    "farmacia": [
        "PCM_BD_farmacia_tedef_2026_septiembre.csv"
    ],
    "eps": [
        "PRES_LIQ_EPS_202508.csv"
    ]
}

ROLE_LABELS = {
    "renipress": "RENIPRESS",
    "c1": "C1 - Producción asistencial",
    "consultas_denuncias": "Consultas y denuncias",
    "farmacia": "Farmacia privada - septiembre 2026",
    "eps": "Liquidaciones EPS"
}


# ------------------------------------------------------------
# 2. IDENTIFICAR EL DATASET SEGÚN EL NOMBRE DEL ARCHIVO
# ------------------------------------------------------------

def get_dataset_role(filename):
    """Identifica los datasets principales de STASYS."""

    original_name = Path(filename).name.strip()
    normalized = original_name.lower()

    # Primero, buscar coincidencias exactas.
    for role, names in EXPECTED_FILES.items():
        if any(normalized == name.lower() for name in names):
            return role

    # Reglas adicionales para nombres ligeramente diferentes.
    name = normalized.replace(" ", "_")

    if "renipress" in name:
        return "renipress"

    if "consultac1" in name or "produccion_asistencial" in name:
        return "c1"

    if "consultas_denuncias" in name:
        return "consultas_denuncias"

    # Regla específica para la nueva base de farmacia.
    if (
        "pcm_bd_farmacia_tedef_2026_septiembre" in name
        or (
            "farmacia" in name
            and "2026" in name
        )
    ):
        return "farmacia"

    if "pres_liq_eps" in name:
        return "eps"

    # No incorporar automáticamente el antiguo TEDEF.
    return "generico"


# ------------------------------------------------------------
# 3. LISTAR ARCHIVOS DE GOOGLE DRIVE
# ------------------------------------------------------------

def list_drive_files(folder_id):
    """Lista archivos admitidos dentro de la carpeta de Drive."""

    found = []
    page_token = None

    while True:
        response = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files(id,name,size,mimeType,modifiedTime)"
            ),
            pageSize=100,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()

        for item in response.get("files", []):
            filename = item.get("name", "")
            extension = Path(filename).suffix.lower()
            mime_type = item.get("mimeType", "")

            # Ignorar carpetas y archivos nativos de Google.
            if mime_type == "application/vnd.google-apps.folder":
                continue

            if mime_type.startswith("application/vnd.google-apps."):
                continue

            if extension in SUPPORTED_EXTENSIONS:
                found.append(item)

        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return sorted(
        found,
        key=lambda item: item.get("name", "").lower()
    )


# ------------------------------------------------------------
# 4. SELECCIONAR UN ARCHIVO POR DATASET
# ------------------------------------------------------------

def select_target_files(drive_files):
    """
    Selecciona como máximo un archivo por dataset.
    Usa primero los nombres oficiales definidos arriba.
    """

    selected = []
    missing = []

    for role, expected_names in EXPECTED_FILES.items():
        match = None

        for expected_name in expected_names:
            matches = [
                item for item in drive_files
                if item.get("name", "").strip().lower()
                == expected_name.lower()
            ]

            if matches:
                matches.sort(
                    key=lambda item: item.get("modifiedTime", ""),
                    reverse=True
                )
                match = matches[0]
                break

        if match:
            selected.append(match)
        else:
            missing.append(ROLE_LABELS[role])

    return selected, missing


# ------------------------------------------------------------
# 5. DESCARGAR ARCHIVOS DE GOOGLE DRIVE
# ------------------------------------------------------------

def download_drive_file(file_id, destination):
    """Descarga un archivo de Drive en bloques."""

    request = drive.files().get_media(
        fileId=file_id,
        supportsAllDrives=True
    )

    with open(destination, "wb") as stream:
        downloader = MediaIoBaseDownload(
            stream,
            request,
            chunksize=16 * 1024 * 1024
        )

        done = False

        while not done:
            status, done = downloader.next_chunk()

            if status:
                print(
                    f"  Progreso: {int(status.progress() * 100)}%",
                    end="\r"
                )

    print("  ✓ Descarga terminada")


# ------------------------------------------------------------
# 6. PREPARAR CARPETA TEMPORAL
# ------------------------------------------------------------

if "RUN_ID" not in globals():
    raise RuntimeError(
        "No existe RUN_ID. Ejecuta primero la celda 03."
    )

if "SUPPORTED_EXTENSIONS" not in globals():
    raise RuntimeError(
        "No existe SUPPORTED_EXTENSIONS. Ejecuta primero la celda 03."
    )

if "DRIVE_FOLDER_ID" not in globals() or not DRIVE_FOLDER_ID:
    DRIVE_FOLDER_ID = "1NSCDPLZlJXPJsddy-D7G64IHHJnR-jwO"

WORK_DIR = Path("/content") / f"stasys_{RUN_ID}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

input_files = []


# ------------------------------------------------------------
# 7. MENÚ PRINCIPAL
# ------------------------------------------------------------

print("=" * 75)
print("STASYS - SELECCIÓN DE DATASETS DEL PIPELINE MEDALLÓN")
print("=" * 75)
print("1. Subir archivos desde mi computadora")
print("2. Elegir archivos de Google Drive")
print("3. Procesar los 5 datasets principales de Google Drive")
print("=" * 75)

opcion = input("Selecciona una opción (1, 2 o 3): ").strip()


# ------------------------------------------------------------
# OPCIÓN 1: SUBIR ARCHIVOS DESDE LA PC
# ------------------------------------------------------------

if opcion == "1":

    print("\nSelecciona los archivos CSV de STASYS.")
    uploaded = colab_files.upload()

    if not uploaded:
        raise RuntimeError("No se seleccionaron archivos.")

    for filename, content in uploaded.items():
        original_name = Path(filename).name
        extension = Path(original_name).suffix.lower()

        if extension not in SUPPORTED_EXTENSIONS:
            print(f"⚠ Formato no admitido: {original_name}")
            continue

        role = get_dataset_role(original_name)

        if role == "generico":
            print(f"⚠ Archivo no reconocido; se omite: {original_name}")
            continue

        if any(
            get_dataset_role(item["name"]) == role
            for item in input_files
        ):
            print(
                f"⚠ Ya se cargó {ROLE_LABELS[role]}. "
                f"Se omite la copia: {original_name}"
            )
            continue

        local_path = WORK_DIR / (
            f"{len(input_files) + 1:03d}_{original_name}"
        )
        local_path.write_bytes(content)

        input_files.append({
            "name": original_name,
            "path": local_path,
            "drive_id": None,
            "modified": "",
            "source": "Carga local"
        })

        print(f"✓ Archivo recibido: {original_name}")


# ------------------------------------------------------------
# OPCIÓN 2: ELEGIR ARCHIVOS DE GOOGLE DRIVE
# ------------------------------------------------------------

elif opcion == "2":

    all_drive_files = list_drive_files(DRIVE_FOLDER_ID)
    drive_files, missing = select_target_files(all_drive_files)

    if missing:
        print("\n⚠ No se encontraron estos datasets:")
        for name in missing:
            print(f"  - {name}")

    if not drive_files:
        raise RuntimeError(
            "No se encontraron los datasets esperados en Drive. "
            "Verifica el ID de la carpeta y los nombres de los archivos."
        )

    print("\nDATASETS DISPONIBLES")
    print("-" * 75)

    for index, item in enumerate(drive_files, start=1):
        role = get_dataset_role(item["name"])
        size = int(item.get("size", 0))

        print(
            f"{index:>2}. [{ROLE_LABELS[role]}] "
            f"{item['name']} ({size:,} bytes)"
        )

    seleccion = input(
        "\nIngresa los números separados por comas (ej. 1,2,3,4,5): "
    ).strip()

    try:
        indices = [
            int(value.strip())
            for value in seleccion.split(",")
            if value.strip()
        ]
    except ValueError:
        raise ValueError("Ingresa números separados por comas.")

    if not indices:
        raise ValueError("No seleccionaste ningún archivo.")

    if len(indices) != len(set(indices)):
        raise ValueError("Hay números repetidos en la selección.")

    if any(i < 1 or i > len(drive_files) for i in indices):
        raise ValueError("Uno de los números está fuera de la lista.")

    selected_files = [drive_files[i - 1] for i in indices]
    roles = [get_dataset_role(item["name"]) for item in selected_files]

    if len(roles) != len(set(roles)):
        raise ValueError(
            "Seleccionaste más de un archivo para el mismo dataset."
        )

    for position, item in enumerate(selected_files, start=1):
        original_name = item["name"]
        local_path = WORK_DIR / (
            f"{position:03d}_{Path(original_name).name}"
        )

        print(f"\nDescargando: {original_name}")
        download_drive_file(item["id"], local_path)

        input_files.append({
            "name": original_name,
            "path": local_path,
            "drive_id": item["id"],
            "modified": item.get("modifiedTime", ""),
            "source": "Google Drive"
        })


# ------------------------------------------------------------
# OPCIÓN 3: PROCESAR LOS 5 DATASETS PRINCIPALES DE DRIVE
# ------------------------------------------------------------

elif opcion == "3":

    all_drive_files = list_drive_files(DRIVE_FOLDER_ID)
    selected_files, missing = select_target_files(all_drive_files)

    if missing:
        print("\n⚠ No se encontraron estos datasets:")
        for name in missing:
            print(f"  - {name}")

    if not selected_files:
        raise RuntimeError(
            "No se encontró ningún dataset principal en Drive. "
            "Comprueba la carpeta y los nombres de los archivos."
        )

    print(
        f"\nSe encontraron {len(selected_files)} "
        "de los 5 datasets principales."
    )

    for index, item in enumerate(selected_files, start=1):
        original_name = item["name"]
        local_path = WORK_DIR / (
            f"{index:03d}_{Path(original_name).name}"
        )

        print(
            f"\n[{index}/{len(selected_files)}] "
            f"Descargando: {original_name}"
        )

        download_drive_file(item["id"], local_path)

        input_files.append({
            "name": original_name,
            "path": local_path,
            "drive_id": item["id"],
            "modified": item.get("modifiedTime", ""),
            "source": "Google Drive"
        })


else:
    raise ValueError(
        "Opción no válida. Ejecuta nuevamente y elige 1, 2 o 3."
    )


# ------------------------------------------------------------
# 8. AUDITORÍA INICIAL DE LOS ARCHIVOS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("AUDITORÍA INICIAL DE DATASETS")
print("=" * 75)

audit_rows = []
audit_errors = []

for item in input_files:
    path = Path(item["path"])

    try:
        iterator = iter_file_chunks(path)
        first_chunk = next(iterator, None)

        if first_chunk is None or first_chunk.empty:
            raise ValueError("El archivo no contiene filas de datos.")

        columns = list(first_chunk.columns)
        role = get_dataset_role(item["name"])

        # Comprobación adicional por estructura de columnas.
        detected_by_columns = classify_dataset(
            item["name"],
            columns
        )

        audit_rows.append({
            "archivo": item["name"],
            "origen": item["source"],
            "tipo_detectado": role,
            "clasificacion_por_columnas": detected_by_columns,
            "columnas": len(columns),
            "filas_primer_bloque": len(first_chunk),
            "bytes": path.stat().st_size,
            "estado": "OK"
        })

        print(f"\n✓ Archivo: {item['name']}")
        print(f"  Origen: {item['source']}")
        print(f"  Dataset: {ROLE_LABELS.get(role, role)}")
        print(f"  Clasificación por columnas: {detected_by_columns}")
        print(f"  Número de columnas: {len(columns)}")
        print(f"  Columnas: {columns}")

        if detected_by_columns != role:
            print(
                "  ⚠ La clasificación por columnas no coincide "
                "con el nombre. Revisar en SILVER."
            )

    except Exception as exc:
        audit_errors.append({
            "archivo": item["name"],
            "error": str(exc)
        })

        print(f"\n✗ Error leyendo {item['name']}: {exc}")


audit_report = pd.DataFrame(audit_rows)

print("\n" + "-" * 75)
print(f"Archivos seleccionados: {len(input_files)}")
print(f"Archivos legibles: {len(audit_rows)}")
print(f"Archivos con errores: {len(audit_errors)}")

if not audit_report.empty:
    display(audit_report)

if audit_errors:
    display(pd.DataFrame(audit_errors))
    raise RuntimeError(
        "La auditoría detectó errores. Corrige los archivos "
        "antes de continuar con la celda 05 - RAW."
    )

if not input_files:
    raise RuntimeError("No hay archivos preparados para procesar.")


# ------------------------------------------------------------
# 9. VALIDAR LOS DATASETS SELECCIONADOS
# ------------------------------------------------------------

selected_roles = {
    get_dataset_role(item["name"])
    for item in input_files
}

missing_roles = set(EXPECTED_FILES.keys()) - selected_roles

if missing_roles:
    print("\n⚠ Esta ejecución no contiene los 5 datasets.")
    print("Datasets faltantes:")

    for role in sorted(missing_roles):
        print(f"  - {ROLE_LABELS[role]}")

    print(
        "\nPuedes continuar con los archivos disponibles, "
        "pero verifica si necesitas ejecutar el pipeline completo."
    )
else:
    print("\n✓ Los cinco datasets principales están seleccionados.")

print("\n✓ Selección y auditoría inicial completadas.")
print("✓ Puedes continuar con la celda 05 - RAW.")
print(f"✓ Carpeta temporal: {WORK_DIR}")


✓ Google Drive API v3 conectada correctamente
STASYS - SELECCIÓN DE DATASETS DEL PIPELINE MEDALLÓN
1. Subir archivos desde mi computadora
2. Elegir archivos de Google Drive
3. Procesar los 5 datasets principales de Google Drive
Selecciona una opción (1, 2 o 3): 3

Se encontraron 5 de los 5 datasets principales.

[1/5] Descargando: RENIPRESS_2026_v6.csv
  ✓ Descarga terminada

[2/5] Descargando: ConsultaC1_Produccion_Asistencial_2015_v1.csv
  ✓ Descarga terminada

[3/5] Descargando: CONSULTAS_DENUNCIAS_2026_v1.csv
  ✓ Descarga terminada

[4/5] Descargando: PCM_BD_farmacia_tedef_2026_septiembre.csv
  ✓ Descarga terminada

[5/5] Descargando: PRES_LIQ_EPS_202508.csv
  ✓ Descarga terminada

AUDITORÍA INICIAL DE DATASETS

✓ Archivo: RENIPRESS_2026_v6.csv
  Origen: Google Drive
  Dataset: RENIPRESS
  Clasificación por columnas: renipress
  Número de columnas: 33
  Columnas: ['INSTITUCION', 'COD_IPRESS', 'NOMBRE', 'CLASIFICACION', 'TIPO_ESTABLECIMIENTO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO'

,archivo,origen,tipo_detectado,clasificacion_por_columnas,columnas,filas_primer_bloque,bytes,estado
0,RENIPRESS_2026_v6.csv,Google Drive,renipress,renipress,33,26850,17685986,OK
1,ConsultaC1_Produccion_Asistencial_2015_v1.csv,Google Drive,c1,c1,14,14175,2011293,OK
2,CONSULTAS_DENUNCIAS_2026_v1.csv,Google Drive,consultas_denuncias,consultas_denuncias,9,50689,7202603,OK
3,PCM_BD_farmacia_tedef_2026_septiembre.csv,Google Drive,farmacia,farmacia,22,100000,343665894,OK
4,PRES_LIQ_EPS_202508.csv,Google Drive,eps,eps,48,100000,403086739,OK



✓ Los cinco datasets principales están seleccionados.

✓ Selección y auditoría inicial completadas.
✓ Puedes continuar con la celda 05 - RAW.
✓ Carpeta temporal: /content/stasys_20261027T151544Z_1870aa


# Capa RAW

In [28]:

# ============================================================
# CELDA 05 - RAW: CONSERVAR LOS ARCHIVOS ORIGINALES EN S3
# Entrada: input_files, generado por la celda 04.
# Salida: raw_results para la celda 06.
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. VALIDAR DEPENDENCIAS
# ------------------------------------------------------------

required_vars = [
    "input_files",
    "RUN_ID",
    "DATE_PATH",
    "S3_BUCKET",
    "s3",
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Faltan variables necesarias: "
        + ", ".join(missing_vars)
        + ". Ejecuta las celdas anteriores en orden."
    )

if not input_files:
    raise RuntimeError(
        "No hay archivos seleccionados. Ejecuta primero la celda 04."
    )

if "slug" not in globals() or "sha256_file" not in globals():
    raise RuntimeError(
        "No están disponibles slug() o sha256_file(). "
        "Ejecuta primero la celda 03."
    )


# ------------------------------------------------------------
# 2. FUNCIÓN PARA SUBIR ARCHIVOS A S3
# ------------------------------------------------------------

def s3_upload_file(
    path,
    key,
    metadata=None,
    content_type="application/octet-stream"
):
    """Sube un archivo local a S3 con metadatos opcionales."""

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"No existe el archivo local: {path}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"El archivo está vacío: {path.name}"
        )

    extra_args = {"ContentType": content_type}

    if metadata:
        extra_args["Metadata"] = {
            str(k).lower().replace("_", "-"): str(v)[:200]
            for k, v in metadata.items()
            if v is not None
        }

    s3.upload_file(
        str(path),
        S3_BUCKET,
        key,
        ExtraArgs=extra_args
    )


# ------------------------------------------------------------
# 3. VALIDAR ARCHIVOS Y NOMBRES
# ------------------------------------------------------------

validated_files = []
seen_names = set()

for item in input_files:
    path = Path(item["path"])
    original_name = Path(item["name"]).name

    if not path.exists():
        raise FileNotFoundError(
            f"No se encuentra el archivo seleccionado: {original_name}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"El archivo está vacío: {original_name}"
        )

    # Evitar que el mismo nombre aparezca más de una vez.
    normalized_name = original_name.strip().lower()

    if normalized_name in seen_names:
        raise ValueError(
            f"Hay archivos duplicados con el mismo nombre: {original_name}"
        )

    seen_names.add(normalized_name)

    validated_files.append({
        **item,
        "name": original_name,
        "path": path
    })


# ------------------------------------------------------------
# 4. SUBIR ORIGINALES A LA CAPA RAW
# ------------------------------------------------------------

raw_results = {}
raw_audit_rows = []
used_dataset_names = set()

for item in validated_files:

    original_name = item["name"]
    local_path = Path(item["path"])

    # Nombre técnico seguro basado en el nombre del archivo.
    base = slug(Path(original_name).stem)
    dataset = base

    # Si los nombres técnicos coinciden, agregar un sufijo estable.
    if dataset in used_dataset_names:
        suffix = sha256_file(local_path)[:8]
        dataset = f"{base}_{suffix}"

    if dataset in used_dataset_names:
        raise ValueError(
            f"No se pudo crear un nombre técnico único para {original_name}"
        )

    used_dataset_names.add(dataset)

    # Calcular integridad del archivo original.
    raw_hash = sha256_file(local_path)
    raw_size = local_path.stat().st_size

    # La estructura conserva cada ejecución sin sobrescribir
    # el histórico de otras ejecuciones.
    raw_key = (
        f"raw/{DATE_PATH}/{dataset}/{RUN_ID}/{original_name}"
    )

    source_origin = item.get("source", "No especificado")

    # Clasificación de origen para el manifiesto.
    role = (
        classify_dataset(original_name, [])
        if "classify_dataset" in globals()
        else "pendiente"
    )

    # Identificación explícita de la nueva fuente de farmacia.
    if original_name.lower() == (
        "pcm_bd_farmacia_tedef_2026_septiembre.csv"
    ).lower():
        role = "farmacia"

    metadata = {
        "dataset": dataset,
        "role": role,
        "sha256": raw_hash,
        "run-id": RUN_ID,
        "source": source_origin,
        "source-file": original_name,
        "size-bytes": raw_size,
    }

    print("\n" + "-" * 75)
    print(f"Archivo original: {original_name}")
    print(f"Dataset técnico: {dataset}")
    print(f"Rol: {role}")
    print(f"Tamaño: {raw_size:,} bytes")
    print(f"SHA-256: {raw_hash}")
    print("Subiendo a S3...")

    s3_upload_file(
        local_path,
        raw_key,
        metadata=metadata,
        content_type="application/octet-stream"
    )

    # Guardar los datos necesarios para SILVER.
    raw_results[dataset] = {
        "dataset": dataset,
        "filename": original_name,
        "path": local_path,
        "raw_key": raw_key,
        "sha256": raw_hash,
        "size_bytes": raw_size,
        "drive_id": item.get("drive_id"),
        "modified": item.get("modified", ""),
        "source": source_origin,
        "role": role,
    }

    raw_audit_rows.append({
        "dataset": dataset,
        "archivo": original_name,
        "rol": role,
        "origen": source_origin,
        "bytes": raw_size,
        "sha256": raw_hash,
        "s3_key": raw_key,
        "estado": "RAW cargado",
    })

    print(f"✓ Original conservado en S3:")
    print(f"  s3://{S3_BUCKET}/{raw_key}")


# ------------------------------------------------------------
# 5. AUDITORÍA DE RAW
# ------------------------------------------------------------

raw_report = pd.DataFrame(raw_audit_rows)

print("\n" + "=" * 75)
print("RESUMEN DE LA CAPA RAW - STASYS")
print("=" * 75)
print(f"RUN_ID: {RUN_ID}")
print(f"Bucket: {S3_BUCKET}")
print(f"Archivos seleccionados: {len(validated_files)}")
print(f"Archivos guardados: {len(raw_results)}")
print(f"Prefijo: raw/{DATE_PATH}/")

display(raw_report)


# ------------------------------------------------------------
# 6. VALIDACIONES FINALES
# ------------------------------------------------------------

if len(raw_results) != len(validated_files):
    raise RuntimeError(
        "La cantidad de archivos guardados no coincide "
        "con la cantidad de archivos seleccionados."
    )

if raw_report["sha256"].isna().any():
    raise RuntimeError(
        "La auditoría RAW contiene hashes vacíos."
    )

print("\n✓ CELDA 05 COMPLETADA")
print("✓ Los originales se conservaron sin transformación.")
print("✓ raw_results está disponible para la celda 06.")



---------------------------------------------------------------------------
Archivo original: RENIPRESS_2026_v6.csv
Dataset técnico: renipress_2026_v6
Rol: renipress
Tamaño: 17,685,986 bytes
SHA-256: 0817a7e15273a2aeff44b502518f73c30268b525eb76159a5455dd1b0269b6f9
Subiendo a S3...
✓ Original conservado en S3:
  s3://staysis-renipress-equipo02-2026/raw/2026/10/27/renipress_2026_v6/20261027T151544Z_1870aa/RENIPRESS_2026_v6.csv

---------------------------------------------------------------------------
Archivo original: ConsultaC1_Produccion_Asistencial_2015_v1.csv
Dataset técnico: consultac1_produccion_asistencial_2015_v1
Rol: c1
Tamaño: 2,011,293 bytes
SHA-256: 7699a0407c9cabc04ec5dfe07332e6bd7ec925c1e874d9e26f33bfa1466ea813
Subiendo a S3...
✓ Original conservado en S3:
  s3://staysis-renipress-equipo02-2026/raw/2026/10/27/consultac1_produccion_asistencial_2015_v1/20261027T151544Z_1870aa/ConsultaC1_Produccion_Asistencial_2015_v1.csv

---------------------------------------------------

,dataset,archivo,rol,origen,bytes,sha256,s3_key,estado
0,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,Google Drive,17685986,0817a7e15273a2aeff44b502518f73c30268b525eb7615...,raw/2026/10/27/renipress_2026_v6/20261027T1515...,RAW cargado
1,consultac1_produccion_asistencial_2015_v1,ConsultaC1_Produccion_Asistencial_2015_v1.csv,c1,Google Drive,2011293,7699a0407c9cabc04ec5dfe07332e6bd7ec925c1e874d9...,raw/2026/10/27/consultac1_produccion_asistenci...,RAW cargado
2,consultas_denuncias_2026_v1,CONSULTAS_DENUNCIAS_2026_v1.csv,consultas_denuncias,Google Drive,7202603,80524d12e668eb5565b3943c9435558fa9c86bff20dbea...,raw/2026/10/27/consultas_denuncias_2026_v1/202...,RAW cargado
3,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,farmacia,Google Drive,343665894,a5445a77f4d27a63008137a61e9e1596ba488927156e62...,raw/2026/10/27/pcm_bd_farmacia_tedef_2026_sept...,RAW cargado
4,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,Google Drive,403086739,358f9eb090ac8ecd347171e0c21f2c64da71d1dfcf7b94...,raw/2026/10/27/pres_liq_eps_202508/20261027T15...,RAW cargado



✓ CELDA 05 COMPLETADA
✓ Los originales se conservaron sin transformación.
✓ raw_results está disponible para la celda 06.


# Capa SILVER

In [29]:

# ============================================================
# CELDA 06 - SILVER: LIMPIEZA, PERFILADO Y EXPORTACIÓN CSV
# Entrada: raw_results, generado por la celda 05.
# Salida: dataset_info, schema_report y silver_report.
#
# Incluye:
# - Procesamiento por bloques.
# - Clasificación específica del dataset de farmacia.
# - Conservación de códigos como texto.
# - Conteo de nulos y valores distintos observados.
# - Exportación CSV a S3.
# ============================================================

from pathlib import Path
from collections import Counter
import pandas as pd
import itertools
import json


# ------------------------------------------------------------
# 1. VALIDAR DEPENDENCIAS
# ------------------------------------------------------------

required_vars = [
    "raw_results",
    "WORK_DIR",
    "CHUNK_SIZE",
    "RUN_ID",
    "DATE_PATH",
    "S3_BUCKET",
    "s3",
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Faltan variables necesarias: "
        + ", ".join(missing_vars)
        + ". Ejecuta las celdas 02, 03, 04 y 05 en orden."
    )

if not raw_results:
    raise RuntimeError(
        "raw_results está vacío. Ejecuta primero la celda 05."
    )

required_functions = [
    "iter_file_chunks",
    "classify_dataset",
    "infer_profile_type",
    "sha256_file",
    "s3_upload_file",
]

missing_functions = [
    name for name in required_functions
    if name not in globals()
]

if missing_functions:
    raise RuntimeError(
        "Faltan funciones auxiliares: "
        + ", ".join(missing_functions)
        + ". Vuelve a ejecutar la celda 03 y luego la celda 05."
    )


# ------------------------------------------------------------
# 2. FUNCIONES AUXILIARES DE SILVER
# ------------------------------------------------------------

def iter_silver_chunks(path, chunk_size=None):
    """
    Lee el archivo por bloques usando el lector común de la celda 03.
    No carga el archivo completo en memoria.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"No existe el archivo RAW local: {path}"
        )

    if path.stat().st_size == 0:
        raise ValueError(
            f"El archivo de entrada está vacío: {path.name}"
        )

    if chunk_size is None:
        chunk_size = CHUNK_SIZE

    yield from iter_file_chunks(
        path,
        chunk_size=chunk_size
    )


def classify_silver_dataset(filename, columns):
    """
    Clasifica el dataset mediante nombre y estructura.
    La farmacia se identifica explícitamente antes de otras reglas.
    """

    name = Path(filename).name.lower()

    if name == "pcm_bd_farmacia_tedef_2026_septiembre.csv":
        return "farmacia"

    normalized_columns = {
        str(column).strip().upper()
        for column in columns
    }

    pharmacy_signature = {
        "VCODIPRESS_FAC",
        "VNUNMVENTPROD_FAR",
        "VMONTUNISIMP_FAR",
        "VMONTPRODFAR_FAR",
        "DFECRECEP_PQ",
    }

    if len(pharmacy_signature.intersection(normalized_columns)) >= 3:
        return "farmacia"

    # Para los otros datasets se reutiliza el clasificador del proyecto.
    return classify_dataset(filename, columns)


def infer_silver_profile_type(column_name, values, role):
    """
    Infiere tipos para el reporte de calidad.
    Los códigos de farmacia se consideran texto, incluso si contienen
    exclusivamente números.
    """

    column = str(column_name).strip().upper()

    pharmacy_code_columns = {
        "VCODIAFAS_FAC",
        "VCODIPRESS_FAC",
        "VTDOCPAGO_FAC",
        "VNDOCPAGO_FAC",
        "VCORRPRESTA_ATE",
        "VCORRPRODFARATE_FAR",
        "VCODCATALOGO_FAR_020",
        "VCODPROD_FAR_020",
        "VCODIGEMID_FAR_020",
        "VTPROD_FAR_003",
        "VCODCATALOGO_FAR_003",
        "VCODPROD_FAR_003",
        "VDIAGASOCPRODFAR_FAR",
        "VPRODEXENIGV_FAR",
        "UBIGEO",
    }

    if role == "farmacia":
        if column in pharmacy_code_columns:
            return "código/texto"

        if column == "DFECRECEP_PQ":
            return "fecha"

        if column in {
            "VNUNMVENTPROD_FAR",
            "VMONTUNISIMP_FAR",
            "VCOPAGOPRODFAR_FAR",
            "VMONTPRODFAR_FAR",
            "VMONTNCUBFAR_FAR",
        }:
            return "numérico (validar formato)"

    return infer_profile_type(column_name, values)


# ------------------------------------------------------------
# 3. DEFINIR COLUMNAS DE LA FARMACIA PARA VALIDACIÓN
# ------------------------------------------------------------

PHARMACY_EXPECTED_COLUMNS = [
    "VCODIAFAS_FAC",
    "VCODIPRESS_FAC",
    "VTDOCPAGO_FAC",
    "VNDOCPAGO_FAC",
    "VCORRPRESTA_ATE",
    "VCORRPRODFARATE_FAR",
    "VCODCATALOGO_FAR_020",
    "VCODPROD_FAR_020",
    "VCODIGEMID_FAR_020",
    "VTPROD_FAR_003",
    "VCODCATALOGO_FAR_003",
    "VCODPROD_FAR_003",
    "VNUNMVENTPROD_FAR",
    "VTUNIDISP_FAR",
    "VMONTUNISIMP_FAR",
    "VCOPAGOPRODFAR_FAR",
    "VMONTPRODFAR_FAR",
    "VMONTNCUBFAR_FAR",
    "VDIAGASOCPRODFAR_FAR",
    "VPRODEXENIGV_FAR",
    "DFECRECEP_PQ",
    "UBIGEO",
]


# ------------------------------------------------------------
# 4. PREPARAR RESULTADOS
# ------------------------------------------------------------

dataset_info = {}
schema_rows = []
silver_audit_rows = []
pharmacy_quality_rows = []

WORK_DIR = Path(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 5. PROCESAR CADA DATASET POR BLOQUES
# ------------------------------------------------------------

for dataset, raw in raw_results.items():

    original_name = raw["filename"]
    source_path = Path(raw["path"])

    print("\n" + "=" * 80)
    print(f"SILVER | {original_name}")
    print("=" * 80)

    chunks = iter_silver_chunks(source_path)

    try:
        first_chunk = next(chunks)
    except StopIteration:
        raise ValueError(
            f"El archivo no contiene datos: {original_name}"
        )

    if first_chunk.empty:
        raise ValueError(
            f"El primer bloque está vacío: {original_name}"
        )

    columns = list(first_chunk.columns)

    role = classify_silver_dataset(
        original_name,
        columns
    )

    if role == "farmacia":
        actual_normalized = {
            str(column).strip().upper()
            for column in columns
        }

        missing_columns = [
            column
            for column in PHARMACY_EXPECTED_COLUMNS
            if column not in actual_normalized
        ]

        extra_columns = [
            column
            for column in columns
            if str(column).strip().upper()
            not in set(PHARMACY_EXPECTED_COLUMNS)
        ]

        print("✓ Dataset identificado como FARMACIA")
        print(f"Columnas detectadas: {len(columns)}")

        if len(columns) != 22 or missing_columns:
            print(
                "⚠ La estructura no coincide exactamente con "
                "las 22 columnas esperadas."
            )

        if missing_columns:
            print("Columnas esperadas no encontradas:")
            for column in missing_columns:
                print(f"  - {column}")

        if extra_columns:
            print("Columnas adicionales detectadas:")
            for column in extra_columns:
                print(f"  - {column}")

    silver_path = WORK_DIR / f"silver_{dataset}.csv"

    # Evitar mezclar resultados de una ejecución anterior.
    if silver_path.exists():
        silver_path.unlink()

    row_count = 0
    null_counts = Counter()
    distinct_samples = {column: set() for column in columns}
    profile_samples = {column: [] for column in columns}

    pharmacy_empty_columns = set(columns)
    pharmacy_ubigeo_missing = 0
    pharmacy_date_missing = 0
    pharmacy_numeric_invalid = Counter()
    pharmacy_date_min = None
    pharmacy_date_max = None

    numeric_columns = {
        "VNUNMVENTPROD_FAR",
        "VMONTUNISIMP_FAR",
        "VCOPAGOPRODFAR_FAR",
        "VMONTPRODFAR_FAR",
        "VMONTNCUBFAR_FAR",
    }

    all_chunks = itertools.chain([first_chunk], chunks)

    for chunk_number, chunk in enumerate(all_chunks, start=1):

        # Todas las partes deben tener los mismos encabezados.
        if list(chunk.columns) != columns:
            raise ValueError(
                f"Los encabezados cambian entre bloques: {original_name}"
            )

        # Se conserva el esquema original y se exporta CSV.
        # No se eliminan columnas ni filas por valores faltantes.
        chunk.to_csv(
            silver_path,
            mode="w" if row_count == 0 else "a",
            index=False,
            header=(row_count == 0),
            encoding="utf-8-sig" if row_count == 0 else "utf-8",
            lineterminator="\n",
        )

        row_count += len(chunk)

        # Perfilado de nulos y valores observados.
        for column in columns:
            series = chunk[column]

            null_counts[column] += int(series.isna().sum())

            # Muestra acotada para la inferencia del tipo.
            remaining = 5000 - len(profile_samples[column])

            if remaining > 0:
                sample_values = (
                    series.dropna()
                    .astype(str)
                    .head(remaining)
                    .tolist()
                )

                profile_samples[column].extend(sample_values)

            # Aproximación de distintos usando una muestra limitada.
            if len(distinct_samples[column]) < 10000:
                distinct_samples[column].update(
                    series.dropna().astype(str).head(10000).tolist()
                )

        # Métricas específicas para farmacia.
        if role == "farmacia":

            # Detectar columnas que están vacías en todos los bloques.
            for column in list(pharmacy_empty_columns):
                if chunk[column].notna().any():
                    pharmacy_empty_columns.discard(column)

            if "UBIGEO" in chunk.columns:
                pharmacy_ubigeo_missing += int(
                    chunk["UBIGEO"].isna().sum()
                )

            if "DFECRECEP_PQ" in chunk.columns:
                raw_dates = chunk["DFECRECEP_PQ"]

                parsed_dates = pd.to_datetime(
                    raw_dates,
                    errors="coerce",
                    dayfirst=True,
                    format="mixed",
                )

                pharmacy_date_missing += int(
                    (raw_dates.notna() & parsed_dates.isna()).sum()
                )

                valid_dates = parsed_dates.dropna()

                if not valid_dates.empty:
                    block_min = valid_dates.min()
                    block_max = valid_dates.max()

                    if (
                        pharmacy_date_min is None
                        or block_min < pharmacy_date_min
                    ):
                        pharmacy_date_min = block_min

                    if (
                        pharmacy_date_max is None
                        or block_max > pharmacy_date_max
                    ):
                        pharmacy_date_max = block_max

            # Evaluar campos numéricos sin modificar los valores exportados.
            for column in numeric_columns.intersection(columns):
                values = chunk[column].dropna().astype(str).str.strip()

                # Aceptar números decimales con punto.
                # También reconocer coma decimal si no hay punto.
                normalized_values = values.where(
                    values.str.contains(".", regex=False),
                    values.str.replace(",", ".", regex=False),
                )

                parsed_numeric = pd.to_numeric(
                    normalized_values,
                    errors="coerce",
                )

                pharmacy_numeric_invalid[column] += int(
                    parsed_numeric.isna().sum()
                )

        if chunk_number == 1 or chunk_number % 10 == 0:
            print(
                f"Bloque {chunk_number}: "
                f"{row_count:,} filas acumuladas"
            )

    if row_count == 0:
        raise ValueError(
            f"No se generaron filas SILVER: {original_name}"
        )

    if not silver_path.exists() or silver_path.stat().st_size == 0:
        raise RuntimeError(
            f"No se creó correctamente el CSV SILVER: {original_name}"
        )


    # --------------------------------------------------------
    # 6. SUBIR CSV SILVER A S3
    # --------------------------------------------------------

    silver_hash = sha256_file(silver_path)

    silver_key = (
        f"silver/cleaned/{dataset}/"
        f"{DATE_PATH}/{RUN_ID}/{dataset}.csv"
    )

    s3_upload_file(
        silver_path,
        silver_key,
        metadata={
            "dataset": dataset,
            "role": role,
            "run-id": RUN_ID,
            "rows": row_count,
            "columns": len(columns),
            "sha256-raw": raw.get("sha256", ""),
            "sha256-silver": silver_hash,
        },
        content_type="text/csv; charset=utf-8",
    )


    # --------------------------------------------------------
    # 7. PERFILADO POR COLUMNA
    # --------------------------------------------------------

    for column in columns:
        sample_values = profile_samples[column]

        schema_rows.append({
            "dataset": dataset,
            "archivo": original_name,
            "rol_detectado": role,
            "columna": column,
            "tipo_inferido": infer_silver_profile_type(
                column,
                sample_values,
                role,
            ),
            "nulos": int(null_counts[column]),
            "nulos_pct": round(
                null_counts[column] * 100 / row_count,
                2,
            ),
            "valores_distintos_muestra": len(
                distinct_samples[column]
            ),
            "filas_evaluadas": row_count,
        })


    # --------------------------------------------------------
    # 8. MÉTRICAS ESPECÍFICAS DE FARMACIA
    # --------------------------------------------------------

    if role == "farmacia":

        pharmacy_quality_rows.append({
            "dataset": dataset,
            "archivo": original_name,
            "filas": row_count,
            "columnas": len(columns),
            "columnas_completamente_vacias": len(
                pharmacy_empty_columns
            ),
            "columnas_vacias": ", ".join(
                sorted(pharmacy_empty_columns)
            ),
            "ubigeo_faltante": pharmacy_ubigeo_missing,
            "ubigeo_faltante_pct": round(
                pharmacy_ubigeo_missing * 100 / row_count,
                2,
            ),
            "fechas_no_parseables": pharmacy_date_missing,
            "fecha_min": (
                str(pharmacy_date_min.date())
                if pharmacy_date_min is not None
                else None
            ),
            "fecha_max": (
                str(pharmacy_date_max.date())
                if pharmacy_date_max is not None
                else None
            ),
        })

        for column, invalid_count in pharmacy_numeric_invalid.items():
            pharmacy_quality_rows.append({
                "dataset": dataset,
                "archivo": original_name,
                "filas": row_count,
                "columnas": len(columns),
                "campo_evaluado": column,
                "valores_no_nulos_no_parseables": invalid_count,
                "nota": (
                    "Métrica de validación; los datos originales "
                    "no se modificaron."
                ),
            })


    # --------------------------------------------------------
    # 9. METADATOS PARA LAS CELDAS GOLD
    # --------------------------------------------------------

    dataset_info[dataset] = {
        **raw,
        "dataset": dataset,
        "filename": original_name,
        "role": role,
        "columns": columns,
        "rows": row_count,
        "silver_key": silver_key,
        "silver_path": silver_path,
        "silver_sha256": silver_hash,
    }

    silver_audit_rows.append({
        "dataset": dataset,
        "archivo": original_name,
        "rol_detectado": role,
        "filas": row_count,
        "columnas": len(columns),
        "nulos_totales": sum(null_counts.values()),
        "sha256_silver": silver_hash,
        "s3_key": silver_key,
        "estado": "SILVER generado",
    })

    print(f"\n✓ Dataset: {dataset}")
    print(f"✓ Rol: {role}")
    print(f"✓ Filas procesadas: {row_count:,}")
    print(f"✓ Columnas: {len(columns)}")
    print(f"✓ Tamaño CSV: {silver_path.stat().st_size:,} bytes")
    print(f"✓ SHA-256 SILVER: {silver_hash}")
    print(f"✓ S3: s3://{S3_BUCKET}/{silver_key}")


# ------------------------------------------------------------
# 10. CREAR REPORTES DE SILVER
# ------------------------------------------------------------

schema_report = pd.DataFrame(schema_rows)
silver_report = pd.DataFrame(silver_audit_rows)
pharmacy_quality_report = pd.DataFrame(pharmacy_quality_rows)

print("\n" + "=" * 80)
print("RESUMEN DE LA CAPA SILVER - STASYS")
print("=" * 80)

print(f"Datasets procesados: {len(dataset_info)}")
print(
    "Filas procesadas en total: "
    f"{sum(info['rows'] for info in dataset_info.values()):,}"
)
print("Formato de salida: CSV UTF-8")

print("\nAuditoría por dataset:")
display(silver_report)

print("\nPerfil de columnas:")
display(schema_report)

if not pharmacy_quality_report.empty:
    print("\nAuditoría de calidad de farmacia:")
    display(pharmacy_quality_report)

# Guardar reportes locales para revisión y trazabilidad.
schema_report_path = WORK_DIR / "silver_schema_report.csv"
silver_report_path = WORK_DIR / "silver_audit_report.csv"

schema_report.to_csv(
    schema_report_path,
    index=False,
    encoding="utf-8-sig",
)

silver_report.to_csv(
    silver_report_path,
    index=False,
    encoding="utf-8-sig",
)

print("\n✓ CELDA 06 COMPLETADA")
print("✓ dataset_info está disponible para GOLD.")
print("✓ schema_report está disponible para las siguientes celdas.")
print("✓ Los CSV SILVER se guardaron en S3.")



SILVER | RENIPRESS_2026_v6.csv
Bloque 1: 26,850 filas acumuladas

✓ Dataset: renipress_2026_v6
✓ Rol: renipress
✓ Filas procesadas: 26,850
✓ Columnas: 33
✓ Tamaño CSV: 15,886,480 bytes
✓ SHA-256 SILVER: 09bca38e813c5f5e5554badd064f0c426fdbaf140c044e4a7fc30f52648e4ec3
✓ S3: s3://staysis-renipress-equipo02-2026/silver/cleaned/renipress_2026_v6/2026/10/27/20261027T151544Z_1870aa/renipress_2026_v6.csv

SILVER | ConsultaC1_Produccion_Asistencial_2015_v1.csv
Bloque 1: 14,175 filas acumuladas

✓ Dataset: consultac1_produccion_asistencial_2015_v1
✓ Rol: c1
✓ Filas procesadas: 14,175
✓ Columnas: 14
✓ Tamaño CSV: 1,602,888 bytes
✓ SHA-256 SILVER: 80e75aeada27827150dc4c48b314fa25a35e8a4fc370a922b6c3613086986299
✓ S3: s3://staysis-renipress-equipo02-2026/silver/cleaned/consultac1_produccion_asistencial_2015_v1/2026/10/27/20261027T151544Z_1870aa/consultac1_produccion_asistencial_2015_v1.csv

SILVER | CONSULTAS_DENUNCIAS_2026_v1.csv
Bloque 1: 50,689 filas acumuladas

✓ Dataset: consultas_denuncias_

,dataset,archivo,rol_detectado,filas,columnas,nulos_totales,sha256_silver,s3_key,estado
0,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,26850,33,183069,09bca38e813c5f5e5554badd064f0c426fdbaf140c044e...,silver/cleaned/renipress_2026_v6/2026/10/27/20...,SILVER generado
1,consultac1_produccion_asistencial_2015_v1,ConsultaC1_Produccion_Asistencial_2015_v1.csv,c1,14175,14,0,80e75aeada27827150dc4c48b314fa25a35e8a4fc370a9...,silver/cleaned/consultac1_produccion_asistenci...,SILVER generado
2,consultas_denuncias_2026_v1,CONSULTAS_DENUNCIAS_2026_v1.csv,consultas_denuncias,50689,9,5062,e58b9f55066f042944b4d4f5c8bfb79dc0470b52e3d1b6...,silver/cleaned/consultas_denuncias_2026_v1/202...,SILVER generado
3,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,farmacia,1607842,22,4905485,6fb90074f626610d295f890ae15d6b0331b007f83a7a14...,silver/cleaned/pcm_bd_farmacia_tedef_2026_sept...,SILVER generado
4,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,739706,48,2310378,60aadb83d43de82c87075aa84cf7eedf39aae2067e7aa8...,silver/cleaned/pres_liq_eps_202508/2026/10/27/...,SILVER generado



Perfil de columnas:


,dataset,archivo,rol_detectado,columna,tipo_inferido,nulos,nulos_pct,valores_distintos_muestra,filas_evaluadas
0,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,INSTITUCION,texto/categoría,0,0.00,12,26850
1,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,COD_IPRESS,código/categoría (texto),0,0.00,10000,26850
2,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,NOMBRE,texto/categoría,17,0.06,9212,26850
3,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,CLASIFICACION,texto/categoría,68,0.25,71,26850
4,renipress_2026_v6,RENIPRESS_2026_v6.csv,renipress,TIPO_ESTABLECIMIENTO,código/categoría (texto),0,0.00,5,26850
...,...,...,...,...,...,...,...,...,...
121,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,TOTAL_GASTOPRESENTADOS,numérico (inferido),0,0.00,11838,739706
122,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,TOTALDEGASTOSDESCONTADOS,numérico (inferido),0,0.00,4808,739706
123,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,TOTALLIQUIDADOSIN_IGV,numérico (inferido),0,0.00,12714,739706
124,pres_liq_eps_202508,PRES_LIQ_EPS_202508.csv,eps,TOTAL_PAGADOSINIGV,numérico (inferido),0,0.00,11769,739706



Auditoría de calidad de farmacia:


,dataset,archivo,filas,columnas,columnas_completamente_vacias,columnas_vacias,ubigeo_faltante,ubigeo_faltante_pct,fechas_no_parseables,fecha_min,fecha_max,campo_evaluado,valores_no_nulos_no_parseables,nota
0,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,3.0,"VCODCATALOGO_FAR_020, VCODIGEMID_FAR_020, VCOD...",81959.0,5.1,0.0,2026-09-01,2026-09-15,NaN,NaN,NaN
1,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VMONTNCUBFAR_FAR,0.0,Métrica de validación; los datos originales no...
2,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VMONTPRODFAR_FAR,0.0,Métrica de validación; los datos originales no...
3,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCOPAGOPRODFAR_FAR,0.0,Métrica de validación; los datos originales no...
4,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VMONTUNISIMP_FAR,0.0,Métrica de validación; los datos originales no...
5,pcm_bd_farmacia_tedef_2026_septiembre,PCM_BD_farmacia_tedef_2026_septiembre.csv,1607842,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VNUNMVENTPROD_FAR,0.0,Métrica de validación; los datos originales no...



✓ CELDA 06 COMPLETADA
✓ dataset_info está disponible para GOLD.
✓ schema_report está disponible para las siguientes celdas.
✓ Los CSV SILVER se guardaron en S3.


# Capa GOLD

In [ ]:

# ============================================================
# CELDA 07 - GOLD: DIMENSIONES DE FECHA, UBIGEO E IPRESS
# Entrada: dataset_info y archivos SILVER.
# Salida: dim_fecha, dim_ubigeo, dim_ipress,
#         map_ubigeo, map_ubicacion y map_ipress.
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata

if "dataset_info" not in globals() or not dataset_info:
    raise RuntimeError("No hay datasets SILVER. Ejecuta primero la celda 06.")

# ------------------------------------------------------------
# 1. Funciones auxiliares de ubicación
# ------------------------------------------------------------

def valid_ubigeo(value):
    """Devuelve un UBIGEO de seis dígitos o None si no es válido."""
    code = normalize_code(value, 6)
    if code is not None and len(code) == 6 and code.isdigit():
        return code
    return None


def geo_source_columns(frame):
    """Reconoce columnas geográficas explícitas."""
    return (
        first_present(frame.columns, ["DEPARTAMENTO", "DEP"]),
        first_present(frame.columns, ["PROVINCIA", "PROV"]),
        first_present(frame.columns, ["DISTRITO", "DIST"]),
    )


def parse_ubicacion_descriptiva(value):
    """
    Interpreta DEPARTAMENTO-PROVINCIA-DISTRITO.
    Divide solo por los dos primeros guiones para preservar
    guiones que formen parte del nombre del distrito.
    """
    if pd.isna(value):
        return None, None, None

    parts = str(value).strip().split("-", 2)

    if len(parts) != 3:
        return None, None, None

    departamento, provincia, distrito = [
        part.strip() for part in parts
    ]

    if not departamento or not provincia or not distrito:
        return None, None, None

    return departamento, provincia, distrito


def normalize_location_text(value):
    """Normaliza texto para comparar ubicaciones descriptivas."""
    if pd.isna(value):
        return ""

    text = str(value).strip().upper()
    text = re.sub(r"\s+", " ", text)
    return text


def location_key(departamento, provincia, distrito):
    """Clave textual estable; no representa un código UBIGEO oficial."""
    return (
        normalize_location_text(departamento)
        + "|"
        + normalize_location_text(provincia)
        + "|"
        + normalize_location_text(distrito)
    )


# ------------------------------------------------------------
# 2. Recorrer SILVER y recopilar fechas, UBIGEO e IPRESS
# ------------------------------------------------------------

geo_records = {}
descriptive_geo_records = {}
ipress_records = {}

min_date = None
max_date = None

geo_priority = {
    "renipress": 4,
    "c1": 3,
    "eps": 2,
    "tedef": 1,
    "consultas_denuncias": 1,
    "generico": 0,
}

for dataset, info in dataset_info.items():
    role = info["role"]
    silver_path = info["silver_path"]

    print(f"Analizando dimensión: {info['filename']} ({role})")

    for chunk in iter_file_chunks(silver_path):

        # A. Fechas de eventos. RENIPRESS es un catálogo.
        if role != "renipress":
            dates = parse_date_values(chunk, role).dropna()
            dates = dates[dates.dt.year.between(1900, 2100)]

            if not dates.empty:
                chunk_min = dates.min().normalize()
                chunk_max = dates.max().normalize()

                if min_date is None or chunk_min < min_date:
                    min_date = chunk_min

                if max_date is None or chunk_max > max_date:
                    max_date = chunk_max

        # B. Ubicaciones descriptivas de Consultas y Denuncias.
        if role == "consultas_denuncias" and "DE_UBIGEO_INST" in chunk.columns:
            for value in chunk["DE_UBIGEO_INST"].dropna().unique():
                departamento, provincia, distrito = (
                    parse_ubicacion_descriptiva(value)
                )

                if departamento is None:
                    continue

                key = location_key(departamento, provincia, distrito)

                candidate = {
                    "ubicacion_clave": "DESC:" + key,
                    "ubigeo_codigo": None,
                    "departamento": departamento,
                    "provincia": provincia,
                    "distrito": distrito,
                    "_priority": geo_priority.get(role, 0),
                }

                previous = descriptive_geo_records.get(key)

                if previous is None:
                    descriptive_geo_records[key] = candidate
                else:
                    # Completar atributos sin sustituir datos existentes.
                    for field in ("departamento", "provincia", "distrito"):
                        if (
                            not previous.get(field)
                            and candidate.get(field)
                        ):
                            previous[field] = candidate[field]

        # C. UBIGEO numéricos procedentes de las demás fuentes.
        geo_col = get_geo_column(chunk.columns, role)

        if geo_col and not (
            role == "consultas_denuncias"
            and geo_col == "DE_UBIGEO_INST"
        ):
            dep_col, prov_col, dist_col = geo_source_columns(chunk)

            fields = [geo_col] + [
                col for col in (dep_col, prov_col, dist_col) if col
            ]

            priority = geo_priority.get(role, 0)

            for record in chunk[fields].drop_duplicates(
                subset=[geo_col]
            ).to_dict("records"):

                code = valid_ubigeo(record.get(geo_col))

                if code is None:
                    continue

                candidate = {
                    "ubicacion_clave": "UBIGEO:" + code,
                    "ubigeo_codigo": code,
                    "departamento": record.get(dep_col) if dep_col else None,
                    "provincia": record.get(prov_col) if prov_col else None,
                    "distrito": record.get(dist_col) if dist_col else None,
                    "_priority": priority,
                }

                previous = geo_records.get(code)

                if previous is None or priority > previous["_priority"]:
                    geo_records[code] = candidate

                elif priority == previous["_priority"]:
                    for field in ("departamento", "provincia", "distrito"):
                        if (
                            pd.isna(previous.get(field))
                            and pd.notna(candidate.get(field))
                        ):
                            previous[field] = candidate[field]

        # D. RENIPRESS alimenta el catálogo de establecimientos.
        if role == "renipress" and "COD_IPRESS" in chunk.columns:

            optional_columns = [
                "NOMBRE", "INSTITUCION", "CLASIFICACION",
                "TIPO_ESTABLECIMIENTO", "CATEGORIA",
                "ESTADO", "SITUACION", "UBIGEO"
            ]

            keep = ["COD_IPRESS"] + [
                col for col in optional_columns if col in chunk.columns
            ]

            for record in chunk[keep].to_dict("records"):
                code = normalize_code(record.get("COD_IPRESS"))

                if not code:
                    continue

                if code not in ipress_records:
                    ipress_records[code] = record


# ------------------------------------------------------------
# 3. DIM_FECHA: calendario diario
# ------------------------------------------------------------

if min_date is not None and max_date is not None:
    calendar_dates = pd.date_range(
        start=min_date, end=max_date, freq="D"
    )
else:
    calendar_dates = pd.DatetimeIndex([])

calendar = pd.DataFrame({"fecha": calendar_dates})

if calendar.empty:
    dim_fecha = pd.DataFrame(columns=[
        "fecha_id", "fecha", "anho", "mes", "dia",
        "trimestre", "dia_semana", "anho_mes"
    ])
else:
    dim_fecha = pd.DataFrame({
        "fecha_id": calendar["fecha"].dt.strftime("%Y%m%d").astype("int64"),
        "fecha": calendar["fecha"].dt.strftime("%Y-%m-%d"),
        "anho": calendar["fecha"].dt.year.astype("int16"),
        "mes": calendar["fecha"].dt.month.astype("int8"),
        "dia": calendar["fecha"].dt.day.astype("int8"),
        "trimestre": calendar["fecha"].dt.quarter.astype("int8"),
        "dia_semana": (calendar["fecha"].dt.dayofweek + 1).astype("int8"),
        "anho_mes": calendar["fecha"].dt.strftime("%Y-%m"),
    })

unknown_date = pd.DataFrame([{
    "fecha_id": 0,
    "fecha": None,
    "anho": 0,
    "mes": 0,
    "dia": 0,
    "trimestre": 0,
    "dia_semana": 0,
    "anho_mes": "DESCONOCIDO",
}])

dim_fecha = pd.concat(
    [unknown_date, dim_fecha], ignore_index=True
)

# ------------------------------------------------------------
# 4. DIM_UBIGEO: códigos oficiales y ubicaciones descriptivas
# ------------------------------------------------------------

geo_rows = []

# Registros que tienen códigos UBIGEO válidos.
for code, record in sorted(geo_records.items()):
    geo_rows.append({
        "ubicacion_clave": "UBIGEO:" + code,
        "ubigeo_codigo": code,
        "departamento": record.get("departamento"),
        "provincia": record.get("provincia"),
        "distrito": record.get("distrito"),
        "tipo_clave": "UBIGEO",
    })

# Ubicaciones descriptivas sin código oficial confirmado.
for key, record in sorted(descriptive_geo_records.items()):
    geo_rows.append({
        "ubicacion_clave": "DESC:" + key,
        "ubigeo_codigo": None,
        "departamento": record.get("departamento"),
        "provincia": record.get("provincia"),
        "distrito": record.get("distrito"),
        "tipo_clave": "DESCRIPTIVA",
    })

dim_ubigeo = pd.DataFrame(
    geo_rows,
    columns=[
        "ubicacion_clave", "ubigeo_codigo", "departamento",
        "provincia", "distrito", "tipo_clave"
    ]
)

# Evitar que claves repetidas generen varias filas.
dim_ubigeo = dim_ubigeo.drop_duplicates(
    subset=["ubicacion_clave"]
).sort_values(
    ["tipo_clave", "ubicacion_clave"]
).reset_index(drop=True)

dim_ubigeo.insert(
    0, "ubigeo_id",
    np.arange(1, len(dim_ubigeo) + 1, dtype="int64")
)

# Mapa de códigos UBIGEO numéricos válidos.
numeric_geo = dim_ubigeo[
    dim_ubigeo["tipo_clave"] == "UBIGEO"
]

map_ubigeo = dict(zip(
    numeric_geo["ubigeo_codigo"],
    numeric_geo["ubigeo_id"]
))
map_ubigeo[None] = 0

# Mapa de nombres descriptivos para que otras celdas puedan
# resolver las ubicaciones de Consultas y Denuncias.
descriptive_geo = dim_ubigeo[
    dim_ubigeo["tipo_clave"] == "DESCRIPTIVA"
]

map_ubicacion = dict(zip(
    descriptive_geo["ubicacion_clave"].str.replace(
        "DESC:", "", regex=False
    ),
    descriptive_geo["ubigeo_id"]
))

# Mapa del valor original completo, normalizado para comparaciones.
map_ubicacion_origen = {}

for key, record in descriptive_geo_records.items():
    map_ubicacion_origen[
        "-".join([
            normalize_location_text(record["departamento"]),
            normalize_location_text(record["provincia"]),
            normalize_location_text(record["distrito"]),
        ])
    ] = map_ubicacion.get(key, 0)

# ------------------------------------------------------------
# 5. DIM_IPRESS: catálogo de establecimientos
# ------------------------------------------------------------

ipress_rows = []

for code, record in sorted(ipress_records.items()):
    ipress_rows.append({
        "codigo_ipress": code,
        "nombre_ipress": record.get("NOMBRE"),
        "institucion": record.get("INSTITUCION"),
        "clasificacion": record.get("CLASIFICACION"),
        "tipo_establecimiento": record.get("TIPO_ESTABLECIMIENTO"),
        "categoria": record.get("CATEGORIA"),
        "estado": record.get("ESTADO"),
        "situacion": record.get("SITUACION"),
        "ubigeo_codigo": valid_ubigeo(record.get("UBIGEO")),
    })

dim_ipress = pd.DataFrame(
    ipress_rows,
    columns=[
        "codigo_ipress", "nombre_ipress", "institucion",
        "clasificacion", "tipo_establecimiento", "categoria",
        "estado", "situacion", "ubigeo_codigo"
    ]
)

dim_ipress.insert(
    0, "ipress_id",
    np.arange(1, len(dim_ipress) + 1, dtype="int64")
)

dim_ipress["ubigeo_id"] = (
    dim_ipress["ubigeo_codigo"]
    .map(map_ubigeo)
    .fillna(0)
    .astype("int64")
)

map_ipress = dict(zip(
    dim_ipress["codigo_ipress"],
    dim_ipress["ipress_id"]
))
map_ipress[None] = 0

# ------------------------------------------------------------
# 6. Validaciones básicas de las dimensiones
# ------------------------------------------------------------

if dim_fecha["fecha_id"].duplicated().any():
    raise ValueError("DIM_FECHA contiene claves duplicadas.")

if dim_ubigeo["ubigeo_id"].duplicated().any():
    raise ValueError("DIM_UBIGEO contiene claves duplicadas.")

if dim_ubigeo["ubicacion_clave"].duplicated().any():
    raise ValueError("DIM_UBIGEO contiene claves de ubicación duplicadas.")

if dim_ubigeo.loc[
    dim_ubigeo["tipo_clave"] == "UBIGEO", "ubigeo_codigo"
].duplicated().any():
    raise ValueError("DIM_UBIGEO contiene códigos UBIGEO duplicados.")

if dim_ipress["ipress_id"].duplicated().any():
    raise ValueError("DIM_IPRESS contiene claves duplicadas.")

if dim_ipress["codigo_ipress"].duplicated().any():
    raise ValueError("DIM_IPRESS contiene códigos IPRESS duplicados.")

print("\n" + "=" * 70)
print("RESUMEN DE DIMENSIONES GOLD")
print("=" * 70)
print(f"dim_fecha:  {len(dim_fecha):,} filas")
print(f"dim_ubigeo: {len(dim_ubigeo):,} filas")
print(f"  - Con UBIGEO numérico: {len(numeric_geo):,}")
print(f"  - Descriptivas: {len(descriptive_geo):,}")
print(f"dim_ipress: {len(dim_ipress):,} filas")

if min_date is not None:
    print(
        f"Rango de fechas: {min_date:%Y-%m-%d} "
        f"a {max_date:%Y-%m-%d}"
    )
else:
    print("⚠ No se identificaron fechas válidas para el calendario.")

print(f"UBIGEO únicos: {numeric_geo['ubigeo_codigo'].nunique():,}")
print(f"IPRESS únicas: {dim_ipress['codigo_ipress'].nunique():,}")
print("Clave 0 en los mapas: valor ausente o no resuelto.")
print("✓ Celda 07 completada.")


Analizando dimensión: RENIPRESS_2026_v6.csv (renipress)
Analizando dimensión: ConsultaC1_Produccion_Asistencial_2015_v1.csv (c1)
Analizando dimensión: CONSULTAS_DENUNCIAS_2026_v1.csv (consultas_denuncias)
Analizando dimensión: PCM_BD_atenciones_tedef_2025_septiembre.csv (tedef)
Analizando dimensión: PRES_LIQ_EPS_202508.csv (eps)

RESUMEN DE DIMENSIONES GOLD
dim_fecha:  4,126 filas
dim_ubigeo: 3,015 filas
  - Con UBIGEO numérico: 2,646
  - Descriptivas: 369
dim_ipress: 26,850 filas
Rango de fechas: 2015-01-01 a 2026-04-17
UBIGEO únicos: 2,646
IPRESS únicas: 26,850
Clave 0 en los mapas: valor ausente o no resuelto.
✓ Celda 07 completada.


In [ ]:

# ============================================================
# CELDA 08 - GOLD: DIMENSIÓN DE FUENTES
# Entrada: dataset_info, generado por la celda 06.
# Salida: dim_fuente y map_source.
# ============================================================

if "dataset_info" not in globals() or not dataset_info:
    raise RuntimeError(
        "No hay datasets procesados. Ejecuta primero la celda 06."
    )

source_rows = []

# Orden estable: los identificadores no dependen del orden
# en que se seleccionaron los archivos.
for dataset_id, (name, info) in enumerate(
    sorted(dataset_info.items(), key=lambda item: item[0].lower()),
    start=1
):
    source_rows.append({
        "dataset_id": dataset_id,
        "dataset": name,
        "rol": info["role"],
        "archivo_origen": info["filename"],
        "filas_silver": int(info["rows"]),
    })

dim_fuente = pd.DataFrame(
    source_rows,
    columns=[
        "dataset_id",
        "dataset",
        "rol",
        "archivo_origen",
        "filas_silver",
    ]
)

# Mapa utilizado posteriormente por la celda 09.
map_source = dict(zip(
    dim_fuente["dataset"],
    dim_fuente["dataset_id"]
))

# Validaciones.
if dim_fuente["dataset_id"].duplicated().any():
    raise ValueError("DIM_FUENTE contiene identificadores duplicados.")

if dim_fuente["dataset"].duplicated().any():
    raise ValueError("DIM_FUENTE contiene datasets duplicados.")

if set(map_source.keys()) != set(dataset_info.keys()):
    raise ValueError("El mapa de fuentes no coincide con dataset_info.")

print("=" * 70)
print("DIMENSIÓN DE FUENTES - STASYS")
print("=" * 70)
print(f"Fuentes registradas: {len(dim_fuente)}")
print("✓ Identificadores asignados en orden determinista.")
display(dim_fuente)

print("✓ Celda 08 completada.")


DIMENSIÓN DE FUENTES - STASYS
Fuentes registradas: 5
✓ Identificadores asignados en orden determinista.


,dataset_id,dataset,rol,archivo_origen,filas_silver
0,1,consultac1_produccion_asistencial_2015_v1,c1,ConsultaC1_Produccion_Asistencial_2015_v1.csv,14175
1,2,consultas_denuncias_2026_v1,consultas_denuncias,CONSULTAS_DENUNCIAS_2026_v1.csv,50689
2,3,pcm_bd_atenciones_tedef_2025_septiembre,tedef,PCM_BD_atenciones_tedef_2025_septiembre.csv,557475
3,4,pres_liq_eps_202508,eps,PRES_LIQ_EPS_202508.csv,739706
4,5,renipress_2026_v6,renipress,RENIPRESS_2026_v6.csv,26850


✓ Celda 08 completada.


In [ ]:

# ============================================================
# CELDA 09 — GOLD: HECHOS POR FUENTE Y TABLA UNIFICADA
# Publica dimensiones y hechos en CSV UTF-8.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

if not dataset_info:
    raise RuntimeError(
        "No hay datasets SILVER. Ejecuta las celdas 06, 07 y 08."
    )

HIST_BASE = f"gold/star_schema/{DATE_PATH}/{RUN_ID}"
LATEST_BASE = "gold/star_schema/latest"
output_manifest = []


def save_csv_table(frame, table_name, folder, latest_folder=None):
    """Guarda una tabla GOLD localmente y la publica en S3."""
    local_path = WORK_DIR / f"{table_name}.csv"

    frame.to_csv(
        local_path,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

    digest = sha256_file(local_path)
    hist_key = f"{HIST_BASE}/{folder}/{table_name}.csv"

    latest_key = (
        f"{LATEST_BASE}/{latest_folder}/{table_name}.csv"
        if latest_folder else None
    )

    metadata = {"run-id": RUN_ID, "sha256": digest}

    s3_upload_file(
        local_path,
        hist_key,
        metadata,
        "text/csv; charset=utf-8"
    )

    if latest_key:
        s3_upload_file(
            local_path,
            latest_key,
            metadata,
            "text/csv; charset=utf-8"
        )

    output_manifest.append({
        "tabla": table_name,
        "formato": "csv",
        "filas": len(frame),
        "columnas": len(frame.columns),
        "sha256": digest,
        "s3_historico": hist_key,
        "s3_latest": latest_key or "",
    })


# ------------------------------------------------------------
# 1. Publicar las dimensiones
# ------------------------------------------------------------

dimensions = {
    "dim_fecha": dim_fecha,
    "dim_ubigeo": dim_ubigeo,
    "dim_ipress": dim_ipress,
    "dim_fuente": dim_fuente,
}

for table_name, frame in dimensions.items():
    save_csv_table(frame, table_name, "dimensions", "dimensions")

print("✓ Dimensiones publicadas en S3.")


# ------------------------------------------------------------
# 2. Preparar el esquema de la tabla de registros unificada
# ------------------------------------------------------------

quality_rows = []

unified_source_columns = sorted({
    f"src_{column.lower()}"
    for info in dataset_info.values()
    for column in info["columns"]
})

index_local = WORK_DIR / "fact_registros.csv"
index_hist_key = f"{HIST_BASE}/facts/fact_registros.csv"
index_latest_key = f"{LATEST_BASE}/facts/fact_registros.csv"

if index_local.exists():
    index_local.unlink()

index_rows = 0
index_columns = 7 + len(unified_source_columns)


# ------------------------------------------------------------
# 3. Generar un hecho detallado para cada dataset
# ------------------------------------------------------------

for dataset, info in dataset_info.items():
    role = info["role"]
    fact_local = WORK_DIR / f"fact_{dataset}.csv"

    if fact_local.exists():
        fact_local.unlink()

    row_offset = 0
    total_rows = 0

    date_missing = 0
    geo_missing = 0
    ipress_missing = 0
    geo_total = 0
    ipress_total = 0

    # Convertir solo columnas identificadas como numéricas
    # en el perfil SILVER, preservando códigos como texto.
    numeric_cols = {
        row["columna"]
        for row in schema_rows
        if row["dataset"] == dataset
        and row["tipo_inferido"] == "numérico (inferido)"
        and not row["columna"].startswith(("COD", "CO_", "UBIGEO"))
        and row["columna"] not in {"ID", "ANHO", "MES"}
        and not row["columna"].endswith("_ID")
        and not any(
            token in row["columna"]
            for token in (
                "CRIPTO", "CRYPTO", "DOCUMENTO", "NUMERO"
            )
        )
    }

    print(f"\nProcesando hecho: fact_{dataset}.csv")

    for chunk in iter_file_chunks(info["silver_path"]):

        # A. Clave de fecha.
        # Cero indica fecha ausente o no resuelta.
        dates = parse_date_values(chunk, role)
        dates = dates.where(
            dates.dt.year.between(1900, 2100)
        )

        date_ids = (
            pd.to_numeric(
                dates.dt.strftime("%Y%m%d"),
                errors="coerce"
            )
            .fillna(0)
            .astype("int64")
        )

        # B. Clave geográfica.
        # Soporta UBIGEO numérico y ubicaciones descriptivas.
        geo_col = get_geo_column(chunk.columns, role)

        if (
            role == "consultas_denuncias"
            and "DE_UBIGEO_INST" in chunk.columns
        ):

            # Separar DEPARTAMENTO-PROVINCIA-DISTRITO.
            # split("-", 2) conserva guiones dentro del distrito.
            def resolve_descriptive_location(value):
                if pd.isna(value):
                    return 0

                value = str(value).strip()

                if not value:
                    return 0

                parts = value.split("-", 2)

                if len(parts) != 3:
                    return 0

                departamento, provincia, distrito = [
                    part.strip() for part in parts
                ]

                if not departamento or not provincia or not distrito:
                    return 0

                # Usar exactamente las funciones y el mapa
                # construidos en la celda 07.
                key = location_key(
                    departamento,
                    provincia,
                    distrito
                )

                return int(map_ubicacion.get(key, 0))

            geo_ids = (
                chunk["DE_UBIGEO_INST"]
                .map(resolve_descriptive_location)
                .astype("int64")
            )

            # Estas ubicaciones no se presentan como códigos
            # UBIGEO oficiales: se resuelven por descripción.
            geo_codes = pd.Series(
                None,
                index=chunk.index,
                dtype="object"
            )

            geo_total += int(geo_ids.ne(0).sum())
            geo_missing += int(geo_ids.eq(0).sum())

        elif geo_col:
            # Fuentes con códigos UBIGEO numéricos.
            geo_codes = chunk[geo_col].map(valid_ubigeo)

            geo_ids = (
                geo_codes.map(map_ubigeo)
                .fillna(0)
                .astype("int64")
            )

            geo_total += int(geo_codes.notna().sum())

            geo_missing += int(
                (geo_codes.notna() & geo_ids.eq(0)).sum()
            )

        else:
            geo_codes = pd.Series(
                None,
                index=chunk.index,
                dtype="object"
            )

            geo_ids = pd.Series(
                0,
                index=chunk.index,
                dtype="int64"
            )

        # C. Clave de IPRESS.
        ipress_col = get_ipress_column(chunk.columns, role)

        if ipress_col:
            ipress_codes = chunk[ipress_col].map(normalize_code)

            ipress_ids = (
                ipress_codes.map(map_ipress)
                .fillna(0)
                .astype("int64")
            )

            ipress_total += int(ipress_codes.notna().sum())

            ipress_missing += int(
                (ipress_codes.notna() & ipress_ids.eq(0)).sum()
            )

        else:
            ipress_codes = pd.Series(
                None,
                index=chunk.index,
                dtype="object"
            )

            ipress_ids = pd.Series(
                0,
                index=chunk.index,
                dtype="int64"
            )

        date_missing += int(date_ids.eq(0).sum())

        # D. Hecho detallado: conserva las columnas de origen.
        fact = chunk.copy()

        for column in numeric_cols.intersection(fact.columns):
            # Convertir a texto antes de reemplazar separadores,
            # para evitar errores si una columna ya es numérica.
            values = fact[column].astype("string")

            fact[column] = pd.to_numeric(
                values.str.replace(",", ".", regex=False),
                errors="coerce"
            )

        fact.insert(
            0,
            "source_row_number",
            np.arange(
                row_offset + 1,
                row_offset + len(chunk) + 1,
                dtype="int64"
            )
        )

        fact.insert(
            0,
            "dataset_id",
            int(map_source[dataset])
        )

        fact.insert(0, "dataset", dataset)

        fact["fecha_id"] = date_ids.to_numpy()
        fact["ubigeo_id"] = geo_ids.to_numpy()
        fact["ipress_id"] = ipress_ids.to_numpy()

        fact.to_csv(
            fact_local,
            mode="a",
            index=False,
            header=(row_offset == 0),
            encoding="utf-8-sig" if row_offset == 0 else "utf-8",
            lineterminator="\n",
        )

        # E. Tabla de consumo unificada.
        # Una fila por fila de origen; no se unen datasets.
        index = pd.DataFrame({
            "dataset_id": np.full(
                len(chunk),
                int(map_source[dataset]),
                dtype="int64"
            ),
            "dataset": dataset,
            "rol": role,
            "source_row_number": np.arange(
                row_offset + 1,
                row_offset + len(chunk) + 1,
                dtype="int64"
            ),
            "fecha_id": date_ids.to_numpy(),
            "ubigeo_id": geo_ids.to_numpy(),
            "ipress_id": ipress_ids.to_numpy(),
        })

        for source_column in unified_source_columns:
            original_column = source_column[4:].upper()

            if original_column in chunk.columns:
                index[source_column] = (
                    chunk[original_column]
                    .astype("string")
                    .reset_index(drop=True)
                )
            else:
                index[source_column] = pd.Series(
                    pd.NA,
                    index=range(len(chunk)),
                    dtype="string"
                )

        index = index[[
            "dataset_id",
            "dataset",
            "rol",
            "source_row_number",
            "fecha_id",
            "ubigeo_id",
            "ipress_id",
            *unified_source_columns
        ]]

        index.to_csv(
            index_local,
            mode="a",
            index=False,
            header=(index_rows == 0),
            encoding="utf-8-sig" if index_rows == 0 else "utf-8",
            lineterminator="\n",
        )

        row_offset += len(chunk)
        total_rows += len(chunk)
        index_rows += len(chunk)

    # F. Validar que GOLD no haya perdido filas.
    if total_rows != info["rows"]:
        raise ValueError(
            f"Control de filas falló en {dataset}: "
            f"SILVER={info['rows']:,}, GOLD={total_rows:,}"
        )

    # G. Publicar el hecho detallado.
    fact_key = f"{HIST_BASE}/facts/fact_{dataset}.csv"
    latest_fact_key = f"{LATEST_BASE}/facts/fact_{dataset}.csv"

    fact_hash = sha256_file(fact_local)

    # Tres claves nuevas + columnas originales.
    fact_columns = len(info["columns"]) + 6

    s3_upload_file(
        fact_local,
        fact_key,
        {"run-id": RUN_ID, "sha256": fact_hash},
        "text/csv; charset=utf-8"
    )

    s3_upload_file(
        fact_local,
        latest_fact_key,
        {"run-id": RUN_ID, "sha256": fact_hash},
        "text/csv; charset=utf-8"
    )

    output_manifest.append({
        "tabla": f"fact_{dataset}",
        "formato": "csv",
        "filas": total_rows,
        "columnas": fact_columns,
        "sha256": fact_hash,
        "s3_historico": fact_key,
        "s3_latest": latest_fact_key,
    })

    quality_rows.append({
        "dataset": dataset,
        "rol": role,
        "filas": total_rows,
        "fechas_sin_clave": date_missing,
        "ubigeos_validos": geo_total,
        "ubigeos_sin_match": geo_missing,
        "ipress_validos": ipress_total,
        "ipress_sin_match": ipress_missing,
        "nota": (
            "Para consultas_denuncias, ubigeos_validos representa "
            "ubicaciones descriptivas resueltas; no códigos oficiales. "
            "Clave 0 = ausente/no resuelta; no se descartan filas."
        ),
    })

    print(
        f"✓ fact_{dataset}: {total_rows:,} filas, "
        f"{fact_columns} columnas"
    )

    if role == "consultas_denuncias":
        print(
            "  Ubicaciones descriptivas resueltas: "
            f"{geo_total:,}"
        )
        print(
            "  Ubicaciones descriptivas sin resolver: "
            f"{geo_missing:,}"
        )


# ------------------------------------------------------------
# 4. Publicar fact_registros.csv
# ------------------------------------------------------------

index_hash = sha256_file(index_local)

for key in (index_hist_key, index_latest_key):
    s3_upload_file(
        index_local,
        key,
        {"run-id": RUN_ID, "rows": index_rows},
        "text/csv; charset=utf-8"
    )

output_manifest.append({
    "tabla": "fact_registros",
    "formato": "csv",
    "filas": index_rows,
    "columnas": index_columns,
    "sha256": index_hash,
    "s3_historico": index_hist_key,
    "s3_latest": index_latest_key,
})

print("\n" + "=" * 75)
print("CELDA 09 — RESUMEN GOLD")
print("=" * 75)
print(f"Datasets procesados: {len(dataset_info)}")
print(f"Filas en fact_registros: {index_rows:,}")
print(f"Tablas registradas: {len(output_manifest)}")
print(f"Histórico: s3://{S3_BUCKET}/{HIST_BASE}/")
print(f"Última versión: s3://{S3_BUCKET}/{LATEST_BASE}/")
print("✓ Hechos y dimensiones publicados.")


✓ Dimensiones publicadas en S3.

Procesando hecho: fact_renipress_2026_v6.csv
✓ fact_renipress_2026_v6: 26,850 filas, 39 columnas

Procesando hecho: fact_consultac1_produccion_asistencial_2015_v1.csv
✓ fact_consultac1_produccion_asistencial_2015_v1: 14,175 filas, 20 columnas

Procesando hecho: fact_consultas_denuncias_2026_v1.csv
✓ fact_consultas_denuncias_2026_v1: 50,689 filas, 15 columnas
  Ubicaciones descriptivas resueltas: 46,805
  Ubicaciones descriptivas sin resolver: 3,884

Procesando hecho: fact_pcm_bd_atenciones_tedef_2025_septiembre.csv
✓ fact_pcm_bd_atenciones_tedef_2025_septiembre: 557,475 filas, 46 columnas

Procesando hecho: fact_pres_liq_eps_202508.csv
✓ fact_pres_liq_eps_202508: 739,706 filas, 54 columnas

CELDA 09 — RESUMEN GOLD
Datasets procesados: 5
Filas en fact_registros: 1,388,895
Tablas registradas: 10
Histórico: s3://staysis-renipress-equipo02-2026/gold/star_schema/2026/10/27/20261027T000553Z_dcf078/
Última versión: s3://staysis-renipress-equipo02-2026/gold/sta

In [ ]:

# ============================================================
# CELDA 10 — CIERRE DE AUDITORÍA Y MANIFIESTO
# Genera reportes de calidad y registra la ejecución.
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

if "output_manifest" not in globals():
    raise RuntimeError("Ejecuta primero la celda 09.")

if "quality_rows" not in globals():
    raise RuntimeError("No existe la auditoría de claves de GOLD.")

# ------------------------------------------------------------
# 1. Crear el reporte de calidad de claves
# ------------------------------------------------------------

quality_report = pd.DataFrame(quality_rows)

# Garantizar que el reporte mantenga su estructura incluso
# si no hay filas en la auditoría.
quality_columns = [
    "dataset", "rol", "filas", "fechas_sin_clave",
    "ubigeos_validos", "ubigeos_sin_match",
    "ipress_validos", "ipress_sin_match", "nota"
]

quality_report = quality_report.reindex(columns=quality_columns)


# ------------------------------------------------------------
# 2. Exportar perfil de columnas y auditoría de claves
# ------------------------------------------------------------

reports = {
    "perfil_columnas": schema_report,
    "auditoria_claves": quality_report,
}

for report_name, frame in reports.items():
    local_path = WORK_DIR / f"{report_name}.csv"

    frame.to_csv(
        local_path,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n",
    )

    report_key = f"{HIST_BASE}/reports/{report_name}.csv"
    report_hash = sha256_file(local_path)

    s3_upload_file(
        local_path,
        report_key,
        {"run-id": RUN_ID, "sha256": report_hash},
        "text/csv; charset=utf-8",
    )

    output_manifest.append({
        "tabla": report_name,
        "formato": "csv",
        "filas": len(frame),
        "columnas": len(frame.columns),
        "sha256": report_hash,
        "s3_historico": report_key,
        "s3_latest": "",
    })

    print(
        f"✓ Reporte generado: {report_name}.csv "
        f"({len(frame):,} filas)"
    )


# ------------------------------------------------------------
# 3. Construir el manifiesto de la ejecución
# ------------------------------------------------------------

manifest = {
    "pipeline": "STASYS_medallon_colab",
    "run_id": RUN_ID,
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "bucket": S3_BUCKET,
    "prefix_historico": HIST_BASE,
    "prefix_latest": LATEST_BASE,
    "formato_silver_gold": "CSV UTF-8",

    "datasets": [
        {
            "dataset": name,
            "archivo_origen": info["filename"],
            "origen": info.get("source", "No especificado"),
            "rol_detectado": info["role"],
            "filas": info["rows"],
            "columnas": info["columns"],
            "sha256_raw": info["sha256"],
            "sha256_silver": info.get("silver_sha256"),
            "raw": info["raw_key"],
            "silver": info["silver_key"],
        }
        for name, info in dataset_info.items()
    ],

    "modelo": {
        "dimensiones": [
            "dim_fecha",
            "dim_ubigeo",
            "dim_ipress",
            "dim_fuente",
        ],
        "hecho_unificado": "facts/fact_registros.csv",
        "hechos_por_dataset": [
            f"facts/fact_{name}.csv"
            for name in dataset_info
        ],
        "regla": (
            "fact_registros conserva una fila por fila de origen "
            "y utiliza columnas src_* para representar los campos "
            "de las diferentes fuentes. No se ejecutan JOIN entre "
            "datasets ni se suman automáticamente métricas "
            "de distinta granularidad."
        ),
    },

    "tablas": output_manifest,
}

manifest_path = WORK_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

manifest_key = f"{HIST_BASE}/manifest.json"

s3_upload_file(
    manifest_path,
    manifest_key,
    {"run-id": RUN_ID},
    "application/json",
)


# ------------------------------------------------------------
# 4. Resumen final de la ejecución
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PIPELINE STASYS - FINALIZADO")
print("=" * 76)

print(f"RUN_ID: {RUN_ID}")
print(f"Datasets procesados: {len(dataset_info)}")
print(
    "Filas consolidadas en fact_registros.csv: "
    f"{sum(info['rows'] for info in dataset_info.values()):,}"
)
print(f"Bucket S3: {S3_BUCKET}")
print(f"Prefijo histórico: s3://{S3_BUCKET}/{HIST_BASE}/")
print(f"Manifiesto: s3://{S3_BUCKET}/{manifest_key}")

print("\nDimensiones:")
for name, frame in dimensions.items():
    print(f"  ✓ {name}: {len(frame):,} filas")

print("\nAuditoría de claves:")
display(quality_report)

print("\nTablas y reportes publicados:")
display(pd.DataFrame(output_manifest))

print("\n✓ Celda 10 completada.")
print("✓ Manifiesto JSON creado y guardado en S3.")


✓ Reporte generado: perfil_columnas.csv (144 filas)
✓ Reporte generado: auditoria_claves.csv (5 filas)

PIPELINE STASYS - FINALIZADO
RUN_ID: 20261026T194251Z_e18b22
Datasets procesados: 5
Filas consolidadas en fact_registros.csv: 1,388,895
Bucket S3: give-me-data
Prefijo histórico: s3://give-me-data/gold/star_schema/2026/10/26/20261026T194251Z_e18b22/
Manifiesto: s3://give-me-data/gold/star_schema/2026/10/26/20261026T194251Z_e18b22/manifest.json

Dimensiones:
  ✓ dim_fecha: 4,126 filas
  ✓ dim_ubigeo: 3,015 filas
  ✓ dim_ipress: 26,850 filas
  ✓ dim_fuente: 5 filas

Auditoría de claves:


,dataset,rol,filas,fechas_sin_clave,ubigeos_validos,ubigeos_sin_match,ipress_validos,ipress_sin_match,nota
0,renipress_2026_v6,renipress,26850,14,26850,0,26850,0,"Para consultas_denuncias, ubigeos_validos repr..."
1,consultac1_produccion_asistencial_2015_v1,c1,14175,0,14175,0,14175,41,"Para consultas_denuncias, ubigeos_validos repr..."
2,consultas_denuncias_2026_v1,consultas_denuncias,50689,0,46805,3884,0,0,"Para consultas_denuncias, ubigeos_validos repr..."
3,pcm_bd_atenciones_tedef_2025_septiembre,tedef,557475,0,555223,0,0,0,"Para consultas_denuncias, ubigeos_validos repr..."
4,pres_liq_eps_202508,eps,739706,0,712330,0,0,0,"Para consultas_denuncias, ubigeos_validos repr..."



Tablas y reportes publicados:


,tabla,formato,filas,columnas,sha256,s3_historico,s3_latest
0,dim_fecha,csv,4126,8,d453399c98f7062f59d687d7fa149a86fa735d6604ebec...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/dimensions/dim_fecha.csv
1,dim_ubigeo,csv,3015,7,fa00b61a49d1a436b362140705ef04e398d7758c3957c4...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/dimensions/dim_ubigeo.csv
2,dim_ipress,csv,26850,11,0596c4bb2aa455a93601de6136095a58477c0a078312a6...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/dimensions/dim_ipress.csv
3,dim_fuente,csv,5,5,3ab18c20dc3590aa2425c770bbab65588f54edac225de4...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/dimensions/dim_fuente.csv
4,fact_renipress_2026_v6,csv,26850,39,3a16ed0fa1386d4412bced4d3058ee11be4b786c1dd8f0...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_renipress_2...
5,fact_consultac1_produccion_asistencial_2015_v1,csv,14175,20,afeaf2de526cde6c092c6c838eb8f3e536e02c385bb1fa...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_consultac1_...
6,fact_consultas_denuncias_2026_v1,csv,50689,15,dff2a462f13513830b8805bac509063524d71f917b61d4...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_consultas_d...
7,fact_pcm_bd_atenciones_tedef_2025_septiembre,csv,557475,46,2ce9cb45177c6a4b8054f93faed17cff58d3b5580bd8d3...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_pcm_bd_aten...
8,fact_pres_liq_eps_202508,csv,739706,54,75f7adc0b22af80d5847af1ff07f18500a81c9eff823e1...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_pres_liq_ep...
9,fact_registros,csv,1388895,145,cecf839fe12bd62d6cfdc726816947f984772f9b566be3...,gold/star_schema/2026/10/26/20261026T194251Z_e...,gold/star_schema/latest/facts/fact_registros.csv



✓ Celda 10 completada.
✓ Manifiesto JSON creado y guardado en S3.


In [ ]:

# ============================================================
# CELDA 11: INSPECCIONAR GOLD - STASYS
# Reutiliza las credenciales de la celda 2
# ============================================================

import boto3
import pandas as pd
from io import BytesIO

# Recuperar las credenciales cargadas por la celda 2
access_key = AWS_ACCESS_KEY_ID
secret_key = AWS_SECRET_ACCESS_KEY
session_token = AWS_SESSION_TOKEN

# Crear una sesión autenticada con esas variables
kwargs = {
    "aws_access_key_id": access_key,
    "aws_secret_access_key": secret_key,
    "region_name": "us-east-1"
}

# Incluir token solo si tiene contenido
if session_token and str(session_token).strip():
    kwargs["aws_session_token"] = session_token

aws_session = boto3.Session(**kwargs)
s3_gold = aws_session.client("s3")

BUCKET = S3_BUCKET
PREFIX = "gold/star_schema/latest/"

# Comprobar acceso al bucket
s3_gold.head_bucket(Bucket=BUCKET)
print(f"Conexión correcta con el bucket: {BUCKET}")

# Listar archivos publicados en GOLD
archivos = []
paginator = s3_gold.get_paginator("list_objects_v2")

for pagina in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for objeto in pagina.get("Contents", []):
        if not objeto["Key"].endswith("/"):
            archivos.append(objeto["Key"])

print(f"\nArchivos encontrados: {len(archivos)}")
for key in archivos:
    print("-", key.replace(PREFIX, ""))

# Inspeccionar CSV: columnas y muestra de cinco filas
print("\n" + "=" * 70)
print("COLUMNAS Y MUESTRAS DE GOLD")
print("=" * 70)

for key in archivos:
    if not key.lower().endswith(".csv"):
        continue

    print(f"\n### TABLA: {key.replace(PREFIX, '')}")

    try:
        response = s3_gold.get_object(Bucket=BUCKET, Key=key)
        df = pd.read_csv(
            BytesIO(response["Body"].read()),
            nrows=5,
            low_memory=False
        )

        print("Columnas:", list(df.columns))
        print(df.to_string(index=False, max_cols=20))

    except Exception as e:
        print(f"Error al leer esta tabla: {type(e).__name__}: {e}")

print("\nInspección de GOLD finalizada.")


Conexión correcta con el bucket: give-me-data

Archivos encontrados: 10
- dimensions/dim_fecha.csv
- dimensions/dim_fuente.csv
- dimensions/dim_ipress.csv
- dimensions/dim_ubigeo.csv
- facts/fact_consultac1_produccion_asistencial_2015_v1.csv
- facts/fact_consultas_denuncias_2026_v1.csv
- facts/fact_pcm_bd_atenciones_tedef_2025_septiembre.csv
- facts/fact_pres_liq_eps_202508.csv
- facts/fact_registros.csv
- facts/fact_renipress_2026_v6.csv

COLUMNAS Y MUESTRAS DE GOLD

### TABLA: dimensions/dim_fecha.csv
Columnas: ['fecha_id', 'fecha', 'anho', 'mes', 'dia', 'trimestre', 'dia_semana', 'anho_mes']
 fecha_id      fecha  anho  mes  dia  trimestre  dia_semana    anho_mes
        0        NaN     0    0    0          0           0 DESCONOCIDO
 20150101 2015-01-01  2015    1    1          1           4     2015-01
 20150102 2015-01-02  2015    1    2          1           5     2015-01
 20150103 2015-01-03  2015    1    3          1           6     2015-01
 20150104 2015-01-04  2015    1    4  

In [ ]:

# ============================================================
# STASYS | GOLD PARA DASHBOARD
# Genera tablas agregadas para el frontend React
# Destino: gold/dashboard/
# ============================================================

import boto3
import pandas as pd
import json
from io import BytesIO
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONEXIÓN A S3
# ------------------------------------------------------------

kwargs = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region_name": "us-east-1"
}

if AWS_SESSION_TOKEN and str(AWS_SESSION_TOKEN).strip():
    kwargs["aws_session_token"] = AWS_SESSION_TOKEN

aws_session = boto3.Session(**kwargs)
s3_gold = aws_session.client("s3")

BUCKET = S3_BUCKET
BASE = "gold/star_schema/latest/"
DESTINO = "gold/dashboard/"

print(f"Bucket: {BUCKET}")
print(f"Destino: s3://{BUCKET}/{DESTINO}")


# ------------------------------------------------------------
# 2. FUNCIONES AUXILIARES
# ------------------------------------------------------------

archivos_generados = []

def leer_gold(ruta):
    respuesta = s3_gold.get_object(
        Bucket=BUCKET,
        Key=BASE + ruta
    )
    return pd.read_csv(
        BytesIO(respuesta["Body"].read()),
        low_memory=False
    )


def publicar_csv(df, nombre):
    if df is None:
        raise ValueError(f"No se puede publicar {nombre}: DataFrame vacío de referencia.")

    buffer = BytesIO()
    df.to_csv(
        buffer,
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

    s3_gold.put_object(
        Bucket=BUCKET,
        Key=DESTINO + nombre,
        Body=buffer.getvalue(),
        ContentType="text/csv; charset=utf-8"
    )

    archivos_generados.append(nombre)
    print(f"✓ {nombre} | Filas: {len(df):,}")


def normalizar_texto(serie):
    return (
        serie.astype("string")
        .str.strip()
        .replace("", pd.NA)
        .fillna("SIN DATO")
    )


def buscar_columna(df, candidatos):
    """Busca una columna ignorando mayúsculas y minúsculas."""
    mapa = {str(c).lower(): c for c in df.columns}
    for candidato in candidatos:
        if candidato.lower() in mapa:
            return mapa[candidato.lower()]
    return None


def resumen_categoria(df, columna, archivo, nombre_cantidad):
    """Publica un conteo por categoría cuando existe la columna."""
    columna_real = buscar_columna(df, [columna])

    if columna_real is None:
        raise KeyError(
            f"No se encontró la columna '{columna}'. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    resultado = (
        df.assign(
            agrupacion=normalizar_texto(df[columna_real])
        )
        .groupby("agrupacion", dropna=False)
        .size()
        .reset_index(name=nombre_cantidad)
        .sort_values(nombre_cantidad, ascending=False)
    )

    publicar_csv(resultado, archivo)


# ------------------------------------------------------------
# 3. FUENTES Y DIMENSIONES GOLD
# ------------------------------------------------------------

dim_fuente = leer_gold("dimensions/dim_fuente.csv")
dim_ipress = leer_gold("dimensions/dim_ipress.csv")
dim_ubigeo = leer_gold("dimensions/dim_ubigeo.csv")

publicar_csv(dim_fuente, "resumen_fuentes.csv")


# ------------------------------------------------------------
# 4. RENIPRESS: TOTAL DE IPRESS
# ------------------------------------------------------------

col_id = buscar_columna(dim_ipress, ["ipress_id"])

if col_id:
    ids = pd.to_numeric(dim_ipress[col_id], errors="coerce")
    ipress_validas = dim_ipress.loc[ids.notna() & ids.gt(0)].copy()
else:
    ipress_validas = dim_ipress.copy()

publicar_csv(
    pd.DataFrame([{
        "indicador": "Establecimientos registrados",
        "valor": len(ipress_validas)
    }]),
    "kpi_ipress.csv"
)

print(f"Total IPRESS consideradas: {len(ipress_validas):,}")


# ------------------------------------------------------------
# 5. RENIPRESS: DISTRIBUCIONES GENERALES
# ------------------------------------------------------------

resumen_categoria(
    ipress_validas, "CLASIFICACION",
    "ipress_por_clasificacion.csv", "cantidad_ipress"
)

resumen_categoria(
    ipress_validas, "CATEGORIA",
    "ipress_por_categoria.csv", "cantidad_ipress"
)

resumen_categoria(
    ipress_validas, "INSTITUCION",
    "ipress_por_institucion.csv", "cantidad_ipress"
)

resumen_categoria(
    ipress_validas, "ESTADO",
    "ipress_por_estado.csv", "cantidad_ipress"
)


# ------------------------------------------------------------
# 6. RENIPRESS: DISTRIBUCIÓN POR DEPARTAMENTO
# Relación entre dim_ipress y dim_ubigeo mediante ubigeo_id.
# ------------------------------------------------------------

col_ipress_ubigeo = buscar_columna(
    ipress_validas, ["ubigeo_id"]
)
col_geo_id = buscar_columna(
    dim_ubigeo, ["ubigeo_id"]
)
col_departamento = buscar_columna(
    dim_ubigeo, ["departamento", "DEPARTAMENTO"]
)

if not col_ipress_ubigeo:
    raise KeyError(
        "dim_ipress no contiene ubigeo_id. "
        "Revisa la estructura de la dimensión en la celda 07."
    )

if not col_geo_id or not col_departamento:
    raise KeyError(
        "dim_ubigeo debe contener ubigeo_id y departamento. "
        f"Columnas encontradas: {list(dim_ubigeo.columns)}"
    )

# Una fila por ID geográfico evita multiplicar IPRESS en el cruce.
mapa_geo = dim_ubigeo[[col_geo_id, col_departamento]].copy()
mapa_geo = mapa_geo.drop_duplicates(subset=[col_geo_id])

ipress_geo = ipress_validas.merge(
    mapa_geo,
    how="left",
    left_on=col_ipress_ubigeo,
    right_on=col_geo_id,
    validate="many_to_one"
)

ipress_geo["agrupacion"] = normalizar_texto(
    ipress_geo[col_departamento]
)

ipress_por_departamento = (
    ipress_geo
    .groupby("agrupacion", dropna=False)
    .size()
    .reset_index(name="cantidad_ipress")
    .sort_values("cantidad_ipress", ascending=False)
)

publicar_csv(
    ipress_por_departamento,
    "ipress_por_departamento.csv"
)

sin_departamento = int(
    ipress_geo[col_departamento].isna().sum()
)

print(
    "IPRESS sin departamento resuelto: "
    f"{sin_departamento:,}"
)


# ------------------------------------------------------------
# 7. CONSULTAS Y DENUNCIAS
# ------------------------------------------------------------

consultas = leer_gold(
    "facts/fact_consultas_denuncias_2026_v1.csv"
)

for columna, archivo in [
    ("DE_TIPO_SOLICITUD", "consultas_por_tipo.csv"),
    ("DE_SOLICITANTE_GENERO", "consultas_por_genero.csv"),
    ("DE_TIPO_INST", "consultas_por_tipo_institucion.csv"),
    ("DE_NATURALEZA_INST", "consultas_por_naturaleza_institucion.csv")
]:
    columna_real = buscar_columna(consultas, [columna])

    if columna_real is None:
        raise KeyError(
            f"Falta {columna} en la tabla de consultas. "
            f"Columnas disponibles: {list(consultas.columns)}"
        )

    resumen = (
        consultas.assign(
            agrupacion=normalizar_texto(consultas[columna_real])
        )
        .groupby("agrupacion", dropna=False)
        .size()
        .reset_index(name="cantidad_registros")
        .sort_values("cantidad_registros", ascending=False)
    )

    publicar_csv(resumen, archivo)


# ------------------------------------------------------------
# 8. PRODUCCIÓN ASISTENCIAL C1
# ------------------------------------------------------------

c1 = leer_gold(
    "facts/fact_consultac1_produccion_asistencial_2015_v1.csv"
)

col_anio = buscar_columna(c1, ["ANHO", "ANIO"])
col_atenciones = buscar_columna(
    c1, ["NRO_TOTAL_ATENCIONES"]
)
col_atendidos = buscar_columna(
    c1, ["NRO_TOTAL_ATENDIDOS"]
)

columnas_medidas = [
    c for c in [col_atenciones, col_atendidos] if c
]

if not col_anio or not columnas_medidas:
    raise KeyError(
        "No se encontraron las columnas de año y medidas de C1. "
        f"Columnas disponibles: {list(c1.columns)}"
    )

c1[col_anio] = pd.to_numeric(c1[col_anio], errors="coerce")

resumen_anual = (
    c1.groupby(col_anio, dropna=False)[columnas_medidas]
    .sum(min_count=1)
    .reset_index()
    .sort_values(col_anio)
)

publicar_csv(resumen_anual, "c1_produccion_anual.csv")

col_c1_departamento = buscar_columna(c1, ["DEPARTAMENTO"])

if col_c1_departamento:
    c1["departamento_dashboard"] = normalizar_texto(
        c1[col_c1_departamento]
    )

    resumen_departamento = (
        c1.groupby("departamento_dashboard", dropna=False)[columnas_medidas]
        .sum(min_count=1)
        .reset_index()
        .sort_values(columnas_medidas[0], ascending=False)
    )

    publicar_csv(
        resumen_departamento,
        "c1_produccion_por_departamento.csv"
    )
else:
    raise KeyError(
        "No se encontró DEPARTAMENTO en el dataset C1."
    )


# ------------------------------------------------------------
# 9. CALIDAD DE DATOS
# ------------------------------------------------------------

col_ubigeo_consultas = buscar_columna(
    consultas, ["DE_UBIGEO_INST"]
)

calidad = pd.DataFrame([
    {
        "dataset": "RENIPRESS",
        "indicador": "Registros procesados",
        "cantidad": len(dim_ipress),
        "observacion": "Filas de dim_ipress"
    },
    {
        "dataset": "RENIPRESS",
        "indicador": "Sin departamento resuelto",
        "cantidad": sin_departamento,
        "observacion": "No se pudo relacionar con dim_ubigeo"
    },
    {
        "dataset": "Consultas y denuncias",
        "indicador": "Registros procesados",
        "cantidad": len(consultas),
        "observacion": "Filas de la tabla de hechos"
    },
    {
        "dataset": "Consultas y denuncias",
        "indicador": "Ubicación sin dato",
        "cantidad": (
            int(consultas[col_ubigeo_consultas].isna().sum())
            if col_ubigeo_consultas else None
        ),
        "observacion": "Valores nulos de ubicación"
    }
])

publicar_csv(calidad, "calidad_datos_dashboard.csv")


# ------------------------------------------------------------
# 10. VALIDAR Y PUBLICAR MANIFIESTO
# ------------------------------------------------------------

tablas_esperadas = [
    "resumen_fuentes.csv",
    "kpi_ipress.csv",
    "ipress_por_departamento.csv",
    "ipress_por_clasificacion.csv",
    "ipress_por_categoria.csv",
    "ipress_por_institucion.csv",
    "ipress_por_estado.csv",
    "consultas_por_tipo.csv",
    "consultas_por_genero.csv",
    "consultas_por_tipo_institucion.csv",
    "consultas_por_naturaleza_institucion.csv",
    "c1_produccion_anual.csv",
    "c1_produccion_por_departamento.csv",
    "calidad_datos_dashboard.csv"
]

faltantes = sorted(set(tablas_esperadas) - set(archivos_generados))

if faltantes:
    raise RuntimeError(
        "No se generaron todos los CSV requeridos: "
        + ", ".join(faltantes)
        + ". No se publicará el manifiesto."
    )

manifiesto = {
    "proyecto": "STASYS",
    "capa": "GOLD",
    "destino": f"s3://{BUCKET}/{DESTINO}",
    "fecha_generacion_utc": datetime.now(timezone.utc).isoformat(),
    "tablas_generadas": archivos_generados
}

s3_gold.put_object(
    Bucket=BUCKET,
    Key=DESTINO + "manifest.json",
    Body=json.dumps(
        manifiesto, ensure_ascii=False, indent=2
    ).encode("utf-8"),
    ContentType="application/json"
)

print("\n✓ GOLD PARA DASHBOARD FINALIZADO")
print(f"CSV generados: {len(archivos_generados)}")
print(f"Destino: s3://{BUCKET}/{DESTINO}")


Bucket: give-me-data
Destino: s3://give-me-data/gold/dashboard/
✓ resumen_fuentes.csv | Filas: 5
✓ kpi_ipress.csv | Filas: 1
Total IPRESS consideradas: 26,850
✓ ipress_por_clasificacion.csv | Filas: 129
✓ ipress_por_categoria.csv | Filas: 11
✓ ipress_por_institucion.csv | Filas: 12
✓ ipress_por_estado.csv | Filas: 1
✓ ipress_por_departamento.csv | Filas: 25
IPRESS sin departamento resuelto: 0
✓ consultas_por_tipo.csv | Filas: 3
✓ consultas_por_genero.csv | Filas: 3
✓ consultas_por_tipo_institucion.csv | Filas: 8
✓ consultas_por_naturaleza_institucion.csv | Filas: 24
✓ c1_produccion_anual.csv | Filas: 1
✓ c1_produccion_por_departamento.csv | Filas: 22
✓ calidad_datos_dashboard.csv | Filas: 4

✓ GOLD PARA DASHBOARD FINALIZADO
CSV generados: 14
Destino: s3://give-me-data/gold/dashboard/
